<h1 align="center">Complejidad y actividad espectral del EEG a través de las etapas del sueño</h1>

<h3 align="center">Trabajo Práctico Final - Introducción a la Ingeniería de Datos</h3>

**Autor:** Ing. Matías R. Pretel  
**Profesor:** Dr. Ing. Nicolás Guillermo Tripp  
**Año:** 2026

# Declaración de uso de IA

Se utilizó Claude Code Opus 5.5 para:
- diseñar la arquitectura del pipeline ETL y los modelos OLTP y dimensional
- generar y revisar el código Python y SQL
- diseñar el dashboard

La IA no generó datos: todos los resultados provienen de la ejecución del pipeline sobre datos públicos.

**Instrucciones de la IA**

Se utilizaron instrucciones para cada etapa de la cadena de procesamiento. Antes de cada sección aparece la instrucción que la generó, con este formato:

> **Prompt N.** *texto de la instrucción*

Los prompts muestran el tipo de instrucción que se le dio a la IA en cada etapa. Sin embargo, los resultados fueron revisados y, cuando hizo falta, adaptados o modificados manualmente.

## Objetivo

Construir un pipeline de datos reproducible que:

1. descarga automáticamente EEG de sueño público (EDF/EDF+);
2. lo organiza según **BIDS**;
3. lo transforma en épocas de 30 s etiquetados con su etapa de sueño;
4. calcula métricas espectrales y de complejidad por época y canal;
5. detecta y documenta datos faltantes, outliers estadísticos y artefactos de señal;
6. carga los resultados en una base **SQLite transaccional normalizada (3NF)**;
7. construye un **Data Warehouse en esquema estrella** y lo consulta con SQL;
8. comunica los resultados con visualizaciones Plotly y un dashboard Streamlit.

## Preguntas de investigación

> **1. ¿Cómo varían la complejidad y la actividad espectral del EEG entre las distintas etapas del sueño?**
>
> **2. ¿Cómo difiere la potencia sigma en N2 entre personas de distintas edades?**

- **Actividad espectral**: potencia absoluta y relativa en las bandas de interés (delta, theta, alfa, sigma y beta).
- **Complejidad**: Sample Entropy (impredecibilidad temporal) y Spectral Entropy (dispersión de la potencia en frecuencia).

**Aclaración:** No se realizaron tests de hipótesis, todos los resultados son descriptivos, por lo que una diferencia observada no implica significancia estadística, y una asociación no implica causalidad.

## Dataset

Se utilizó el dataset público **Sleep-EDF Expanded** (Kemp et al., 2000; PhysioNet, DOI [10.13026/C2X676](https://doi.org/10.13026/C2X676)),
subconjunto *sleep-cassette* (SC): polisomnografías ambulatorias de ~20 h de personas sanas sin medicación.

A continuación se describe el contenido del dataset.

| Elemento | Descripción |
|---|---|
| `SC4ssN?0-PSG.edf` | Polisomnografía: EEG Fpz-Cz y Pz-Oz (100 Hz), EOG (100 Hz), EMG, respiración, temperatura (1 Hz) |
| `SC4ssN??-Hypnogram.edf` | Etapas de sueño anotadas por expertos (reglas de Rechtschaffen & Kales) en segmentos múltiplos de 30 s |
| `SC-subjects.xls` | Edad, sexo y hora de apagado de luces por sujeto y noche |
| `SHA256SUMS.txt` | Lista oficial de archivos con sus hashes |

`ss` = sujeto, `N` = noche. El sufijo del hipnograma (p. ej. `EC`, `EH`) **difiere** del del PSG (`E0`), por lo que
el emparejamiento se hace por (sujeto, noche) y no por nombre de archivo.

**Selección de sujetos.** Se procesan **18 sujetos × 2 noches**, con 3 mujeres y 3 hombres por grupo etario.

Los 18 sujetos forman una muestra estratificada: 3 mujeres y 3 hombres en cada grupo etario (<40, 40–64 y 65+).
La selección usa solo edad, sexo y cantidad de noches, sin mirar las señales: dentro de cada sexo se toman edades
repartidas por todo el rango, y cada grupo se completa con las edades más separadas entre sí.

| Grupo etario | Sexo | ID del sujeto | Edad (años) |
|---|---|---|---|
| <40 | Mujeres | 5 · 6 · 4 | 28 · 31 · 34 |
| <40 | Hombres | 10 · 14 · 16 | 26 · 27 · 32 |
| 40–64 | Mujeres | 20 · 24 · 22 | 51 · 54 · 56 |
| 40–64 | Hombres | 38 · 31 · 33 | 51 · 54 · 60 |
| 65+ | Mujeres | 40 · 65 · 61 | 67 · 88 · 101 |
| 65+ | Hombres | 50 · 76 · 73 | 71 · 90 · 97 |

El ID es el número de sujeto de Sleep-EDF: por ejemplo, el sujeto 4 corresponde a los archivos `SC4041…` (noche 1)
y `SC4042…` (noche 2). Dentro de cada fila, los sujetos están ordenados por edad.

El dataset no tiene sujetos de 35 a 49 años, ni de 61 a 65.

**Decisiones tomadas durante el trabajo:**

| Decisión | Motivo |
|---|---|
| Recortar la vigilia | Sin el recorte, W domina el análisis |
| Muestra estratificada por sexo y edad | Para comparar edades hacen falta grupos del mismo tamaño y con ambos sexos |
| Limitar la noche a la franja 21:00–12:00 | Las siestas diurnas estiraban la "noche" a casi 24 h |
| Reemplazar el umbral de amplitud por saturación | El umbral de 500 µV no se podía alcanzar con el rango físico del EDF |
| Calcular Tukey dentro de cada noche | Con la referencia global, la regla marcaba personas y no épocas raras |
| Usar 3 dimensiones en el almacén | Cada registro pertenece a un único sujeto, así que grupo etario → sujeto → registro es una jerarquía dentro de la dimensión registro. Con canal y etapa quedan 3 dimensiones, que forman el cubo |


## Arquitectura general del proyecto

<h1 style="color: red;">Revisar</h1>

A continuación se describen los módulos utilizados en este proyecto.

| Módulo | Responsabilidad |
|---|---|
| `config.py` | Rutas relativas, selección de sujetos, parámetros metodológicos y umbrales |
| `extract.py` | Catálogo, selección, descarga validada y manifest |
| `bids.py` | Conversión a BIDS y metadatos |
| `preprocessing.py` | Lectura EDF, validación, epoching, recorte de vigilia, filtrado |
| `features.py` | Métricas espectrales y de complejidad |
| `quality.py` | Artefactos, outliers estadísticos, reporte de faltantes |
| `outliers.py` | Diagnóstico exploratorio de outliers: reglas univariadas, detectores multivariados y sensibilidad |
| `pipeline.py` | Orquestación del Transform con caché por registro |
| `database_oltp.py` / `database_dw.py` | Bases SQLite transaccional y analítica |
| `queries.py` / `plots.py` | Consultas SQL y figuras (compartidas con el dashboard) |
| `validation.py` | Checks automáticos con mensajes comprensibles |
| `app.py` | Dashboard Streamlit |

## Configuración

La primera celda prepara el entorno. En **Google Colab** clona el repositorio e instala las dependencias; localmente
solo agrega la raíz del proyecto al `sys.path`. Todas las rutas se calculan relativas a la raíz con `pathlib`.

`QUICK_RUN = True` procesa 2 sujetos (≈5 min); `False` usa la configuración completa (18 sujetos × 2 noches,
≈40–45 min la primera vez; las ejecuciones siguientes reutilizan el caché).

> **Prompt 0.** *Tomá el rol de ingeniero de datos con experiencia en EEG del sueño. La idea es armar un pipeline reproducible con el dataset público Sleep-EDF Expanded de PhysioNet. El pipeline de procesamiento será el siguiente:*
>
> - Descargar por código los registros del dataset
> - Extraer las señales de los registros y separarlas en épocas de 30 s con su etapa de sueño
> - Calcular métricas espectrales y de complejidad
> - Hacer un control de calidad de las señales
> - Cargar las métricas en una base de datos SQLite normalizada, y crear un almacén en esquema estrella y por último un dashboard.
>
> Consideraciones:
> - Todo tiene que correr desde este notebook, en Colab o de forma local, sin rutas locales ni carga manual, y el repositorio va a mantener solo los directorios "data", "notebooks", "src" y un documento de texto "README.md".
> - No tomes ninguna acción por el momento.

> **Prompt 1.** *El notebook tiene que correr tanto en Colab como en mi compu, sin que haya que tocar nada. Si está en Colab, que clone el repo de GitHub, instale lo que está en "requirements.txt" y se pare en la carpeta del proyecto. Si es local, que encuentre solo la raíz del proyecto, aunque lo abra desde la carpeta "notebooks". No usar rutas fijas. Agregá un flag para hacer una prueba rápida solo con 2 sujetos.*

**Nota:** El archivo con las versiones de las librerías utilizadas "requirements.txt" se fué actualizando a medida que se fue avanzando con el proyecto.

In [ ]:
import os, sys, subprocess
from pathlib import Path

QUICK_RUN = True  # True: 2 sujetos para una prueba rápida

# Repositorio público del trabajo: en Colab se clona desde acá.
REPO_URL = "https://github.com/Mpretel/IID_PF_MP.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content") / Path(REPO_URL).stem
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    PROJECT_ROOT = repo_dir
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Raíz del proyecto:", PROJECT_ROOT)

Raíz del proyecto: C:\workspace\IID_TPF


> **Prompt 2.** *Ahora armá una celda que deje el entorno listo: que los mensajes de avance se vean pero sin el ruido de MNE, que las tablas se muestren completas y que los resultados con azar sean siempre los mismos. Que tome los sujetos de la configuración, respetando el modo de prueba rápida. Y al final mostrame un resumen con los parámetros, así queda registrado con qué se corrió todo.*

In [ ]:
import json, logging, random, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Parámetros del proyecto. Los demás módulos de src/ se importan en la sección que los usa.
from src import config

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("mne").setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 20)  # tablas largas: se muestran las primeras y últimas 5 filas

random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)

# Selección de sujetos: lista explícita de config (o los primeros N si no hay lista).
if config.SUBJECT_IDS:
    SELECTION = {"subject_ids": config.SUBJECT_IDS[:2] if QUICK_RUN else config.SUBJECT_IDS}
else:
    SELECTION = {"n_subjects": 2 if QUICK_RUN else config.N_SUBJECTS}

print("Selección:", SELECTION, "| noches:", config.NIGHTS)
print("Canales EEG:", config.EEG_CHANNELS)
print("Época:", config.EPOCH_DURATION_S, "s | filtro:", config.L_FREQ_HZ, "-", config.H_FREQ_HZ, "Hz",
      "| vigilia conservada:", config.WAKE_PADDING_MIN, "min")
print("Bandas:", config.FREQ_BANDS)
print("Umbrales de artefacto: ≥", config.ARTIFACT_MIN_CLIPPED_SAMPLES, "muestras saturadas; SD <", config.ARTIFACT_MIN_STD_UV, "µV")

Selección: {'subject_ids': [4, 5, 6, 10, 14, 16, 20, 22, 24, 31, 33, 38, 40, 50, 61, 65, 73, 76]} | noches: (1, 2)
Canales EEG: ['EEG Fpz-Cz', 'EEG Pz-Oz']
Época: 30.0 s | filtro: 0.5 - 30.0 Hz | vigilia conservada: 30 min
Bandas: {'delta': (0.5, 4.0), 'theta': (4.0, 8.0), 'alpha': (8.0, 12.0), 'sigma': (12.0, 16.0), 'beta': (16.0, 30.0)}
Umbrales de artefacto: ≥ 10 muestras saturadas; SD < 0.5 µV


> **Prompt 3.** *Armá la extracción de Sleep-EDF Expanded por código, sin carga manual ni rutas locales: PSG e hipnogramas de un grupo de sujetos, 2 noches cada uno. Verificá cada archivo con su checksum y guardá un manifest.*

> **Prompt 4.** *Para poder comparar edades, armá una muestra estratificada de 18 sujetos: 3 mujeres y 3 hombres en cada grupo etario. Elegí los sujetos con una regla que use solo edad, sexo y cantidad de noches, sin mirar sus señales.*

## Extract

```
PhysioNet
    ↓
EDF PSG + Hypnogram
    ↓
data/raw/
```

1. Se descarga `SHA256SUMS.txt`, la lista oficial de archivos del dataset con sus hashes.
2. Con ella se arma el **catálogo** de registros SC, emparejando PSG e hipnograma por (sujeto, noche).
3. Se seleccionan los sujetos de `config.SUBJECT_IDS` (o los primeros `N_SUBJECTS`) y las noches de `config.NIGHTS`.
4. Cada archivo se descarga **solo si no existe o no es válido**; la validación exige tamaño mínimo y que el SHA256
   coincida con el oficial. La descarga es atómica (`.part` + renombrado), por lo que una interrupción no deja
   archivos corruptos.
5. Se registra todo en `data/raw/download_manifest.csv` (archivo, tamaño, hash, estado, URL de origen).

**Fuente**: se usa primero el *mirror* oficial de PhysioNet en AWS Open Data (~50× más rápido en las pruebas) y, si
falla, `physionet.org`. Como cada archivo se verifica contra los hashes oficiales, la fuente no altera los datos.

In [ ]:
from src import extract, validation


def show_checks(lines):
    """Muestra el resultado de un bloque de validaciones (cada una lanza ValidationError si falla)."""
    for line in lines:
        print("  ✔", line)


catalog = extract.build_catalog(extract.fetch_checksums())
print(f"Catálogo sleep-cassette: {len(catalog)} registros de {catalog['subject'].nunique()} sujetos")
print("Sujetos con una sola noche:", sorted(catalog.groupby("subject")["night"].count().loc[lambda s: s < 2].index))

manifest = extract.run_extract(**SELECTION)
display(manifest[["recording_id", "kind", "file", "size_bytes", "status"]])
show_checks(validation.validate_downloads(manifest))

INFO src.extract: Registros seleccionados: 36 (18 sujetos)


Catálogo sleep-cassette: 153 registros de 78 sujetos
Sujetos con una sola noche: [13, 36, 52]


INFO src.extract: Archivos: 72 (72 en caché, 0 descargados)


,recording_id,kind,file,size_bytes,status
0,SC4041,PSG,data/raw/sleep-cassette/SC4041E0-PSG.edf,46878848,cached
1,SC4041,Hypnogram,data/raw/sleep-cassette/SC4041EC-Hypnogram.edf,4830,cached
2,SC4042,PSG,data/raw/sleep-cassette/SC4042E0-PSG.edf,50928128,cached
3,SC4042,Hypnogram,data/raw/sleep-cassette/SC4042EC-Hypnogram.edf,5276,cached
4,SC4051,PSG,data/raw/sleep-cassette/SC4051E0-PSG.edf,49651328,cached
...,...,...,...,...,...
67,SC4732,Hypnogram,data/raw/sleep-cassette/SC4732EJ-Hypnogram.edf,6332,cached
68,SC4761,PSG,data/raw/sleep-cassette/SC4761E0-PSG.edf,47973248,cached
69,SC4761,Hypnogram,data/raw/sleep-cassette/SC4761EP-Hypnogram.edf,4180,cached
70,SC4762,PSG,data/raw/sleep-cassette/SC4762E0-PSG.edf,51256448,cached


  ✔ 72 archivos presentes en data/raw
  ✔ 36 registros con PSG + Hypnogram


> **Prompt 5.** *Organizá los registros en BIDS con MNE-BIDS, sin tocar los datos crudos.*

## Organización BIDS

### EDF y BIDS no son lo mismo

- **EDF/EDF+** (*European Data Format*) es un **formato de archivo**: define cómo se codifican en binario las señales
  de un registro (header con canales, unidades, rango físico/digital, frecuencia de muestreo, y luego los bloques de
  datos). EDF+ agrega anotaciones, que en Sleep-EDF contienen el hipnograma.
- **BIDS** (*Brain Imaging Data Structure*) es una **convención de organización** de un *dataset* completo: nombres de
  carpetas y archivos (`sub-XX/ses-N/eeg/...`), y metadatos legibles en TSV/JSON (`participants.tsv`,
  `channels.tsv`, `events.tsv`, `dataset_description.json`). BIDS no reemplaza a EDF: **lo contiene** (EEG-BIDS
  admite EDF como formato de datos).

### Por qué BIDS mejora la reproducibilidad y la interoperabilidad

- Cualquier herramienta compatible (MNE-BIDS, EEGLAB, validadores BIDS) sabe dónde está cada dato y qué significa,
  sin código ad hoc para los nombres de Sleep-EDF.
- Los metadatos (sujetos, canales, eventos) quedan en texto plano, versionables y legibles por humanos.
- Separa claramente datos crudos, derivados y descripción del dataset.

### Decisiones

- `data/raw/` **no se modifica**. MNE-BIDS copia el EDF a `data/bids/` sin reescribir la señal (se verifica byte a
  byte durante el desarrollo).
- La sesión BIDS es la noche (`ses-1`, `ses-2`) y la tarea es `sleep`.
- Las etapas de sueño se exportan como eventos en `*_events.tsv`.
- `participants.tsv` se completa con edad y sexo de `SC-subjects.xls`, porque el EDF de Sleep-EDF no los incluye.
- `channels.tsv` se corrige con los valores del header EDF. MNE remuestrea internamente todos los canales a 100 Hz,
  pero el archivo real tiene EMG, respiración y temperatura a 1 Hz; los metadatos deben describir el archivo, no la
  representación en memoria.

In [ ]:
from src import bids

bids_index = bids.run_bids(manifest)
display(bids_index)

print("Estructura (primer sujeto):")
first = sorted(config.BIDS_DIR.glob("sub-*"))[0]
for path in sorted(first.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(config.BIDS_DIR).as_posix())

display(pd.read_csv(config.BIDS_DIR / "participants.tsv", sep="\t"))
channels_tsv = next(first.rglob("*_channels.tsv"))
display(pd.read_csv(channels_tsv, sep="\t")[["name", "type", "units", "low_cutoff", "high_cutoff", "sampling_frequency"]])

INFO src.bids: BIDS: 36 registros de 18 sujetos en data/bids


,recording_id,subject,night,bids_file
0,SC4041,4,1,data/bids/sub-04/ses-1/eeg/sub-04_ses-1_task-s...
1,SC4042,4,2,data/bids/sub-04/ses-2/eeg/sub-04_ses-2_task-s...
2,SC4051,5,1,data/bids/sub-05/ses-1/eeg/sub-05_ses-1_task-s...
3,SC4052,5,2,data/bids/sub-05/ses-2/eeg/sub-05_ses-2_task-s...
4,SC4061,6,1,data/bids/sub-06/ses-1/eeg/sub-06_ses-1_task-s...
...,...,...,...,...
31,SC4652,65,2,data/bids/sub-65/ses-2/eeg/sub-65_ses-2_task-s...
32,SC4731,73,1,data/bids/sub-73/ses-1/eeg/sub-73_ses-1_task-s...
33,SC4732,73,2,data/bids/sub-73/ses-2/eeg/sub-73_ses-2_task-s...
34,SC4761,76,1,data/bids/sub-76/ses-1/eeg/sub-76_ses-1_task-s...


Estructura (primer sujeto):
   sub-04/ses-1/eeg/sub-04_ses-1_task-sleep_channels.tsv
   sub-04/ses-1/eeg/sub-04_ses-1_task-sleep_eeg.edf
   sub-04/ses-1/eeg/sub-04_ses-1_task-sleep_eeg.json
   sub-04/ses-1/eeg/sub-04_ses-1_task-sleep_events.json
   sub-04/ses-1/eeg/sub-04_ses-1_task-sleep_events.tsv
   sub-04/ses-1/sub-04_ses-1_scans.tsv
   sub-04/ses-2/eeg/sub-04_ses-2_task-sleep_channels.tsv
   sub-04/ses-2/eeg/sub-04_ses-2_task-sleep_eeg.edf
   sub-04/ses-2/eeg/sub-04_ses-2_task-sleep_eeg.json
   sub-04/ses-2/eeg/sub-04_ses-2_task-sleep_events.json
   sub-04/ses-2/eeg/sub-04_ses-2_task-sleep_events.tsv
   sub-04/ses-2/sub-04_ses-2_scans.tsv


,participant_id,age,sex
0,sub-04,34,F
1,sub-05,28,F
2,sub-06,31,F
3,sub-10,26,M
4,sub-14,27,M
5,sub-16,32,M
6,sub-20,51,F
7,sub-22,56,F
8,sub-24,54,F
9,sub-31,54,M


,name,type,units,low_cutoff,high_cutoff,sampling_frequency
0,EEG Fpz-Cz,EEG,µV,0.50,100.0,100.0
1,EEG Pz-Oz,EEG,µV,0.50,100.0,100.0
2,EOG horizontal,EOG,µV,0.50,100.0,100.0
3,Resp oro-nasal,RESP,NaN,0.03,0.9,1.0
4,EMG submental,EMG,µV,16.00,0.7,1.0
5,Temp rectal,TEMP,oC,NaN,NaN,1.0
6,Event marker,MISC,NaN,NaN,NaN,1.0


> **Prompt 6.** *Antes de transformar, inspeccioná los headers: canales, frecuencia de muestreo y unidades. Cargá solo los dos canales EEG y elegí tipos de datos que no desperdicien memoria.*

## Exploración inicial

Antes de procesar se inspecciona **cada** registro sin suponer valores: canales, frecuencia de muestreo y unidades
(leídas del header EDF), duración, anotaciones y coherencia temporal entre PSG e hipnograma. `validate_recording`
detiene el pipeline con un mensaje claro si falta un canal, la frecuencia o la unidad no son las esperadas, el PSG y
el hipnograma no comparten inicio, o aparece una etiqueta de etapa desconocida.

In [ ]:
from src import preprocessing

files = manifest.pivot(index=["recording_id", "subject", "night"], columns="kind", values="file").reset_index()
rows, warnings_found = [], []
for rec in files.itertuples(index=False):
    info = preprocessing.inspect_recording(config.PROJECT_ROOT / rec.PSG, config.PROJECT_ROOT / rec.Hypnogram)
    warnings_found += preprocessing.validate_recording(info)
    rows.append({
        "recording_id": rec.recording_id,
        "inicio": info["start_datetime"],
        "duración PSG [h]": round(info["psg_duration_s"] / 3600, 2),
        "fin hipnograma [h]": round(info["hypnogram_end_s"] / 3600, 2),
        "fs EEG [Hz]": sorted(set(info["eeg_sfreq_hz"].values())),
        "unidades EEG": sorted(set(info["eeg_units"].values())),
        "etiquetas": ", ".join(f"{k.replace('Sleep stage ', '')}:{v}" for k, v in sorted(info["annotation_counts"].items())),
    })
display(pd.DataFrame(rows))
print("Canales del PSG:", info["channels"])
print("Prefiltro de adquisición EEG:", info["eeg_prefilter"])
print(f"Advertencias ({len(warnings_found)}):")
for w in warnings_found[:3]:
    print("  -", w)
if len(warnings_found) > 3:
    print(f"  ... y {len(warnings_found) - 3} más")

,recording_id,inicio,duración PSG [h],fin hipnograma [h],fs EEG [Hz],unidades EEG,etiquetas
0,SC4041,1989-04-24 16:50:00,21.42,24.0,[100.0],[uV],"Movement time:1, 1:43, 2:47, 3:15, ?:1, R:24, ..."
1,SC4042,1989-04-25 15:04:00,23.27,24.0,[100.0],[uV],"Movement time:4, 1:51, 2:48, 3:22, 4:3, ?:1, R..."
2,SC4051,1989-04-26 16:25:00,22.68,24.0,[100.0],[uV],"1:19, 2:45, 3:39, 4:11, ?:1, R:3, W:12"
3,SC4052,1989-04-27 15:42:00,23.38,24.0,[100.0],[uV],"Movement time:2, 1:43, 2:32, 3:14, 4:6, ?:1, R..."
4,SC4061,1989-05-08 16:35:00,23.08,24.0,[100.0],[uV],"1:18, 2:27, 3:11, 4:4, ?:1, R:4, W:13"
...,...,...,...,...,...,...,...
31,SC4652,1989-11-07 14:40:00,23.67,24.0,[100.0],[uV],"Movement time:1, 1:103, 2:125, 3:77, 4:21, ?:2..."
32,SC4731,1989-10-11 14:55:00,23.17,24.0,[100.0],[uV],"1:60, 2:30, ?:1, R:7, W:35"
33,SC4732,1989-10-12 14:30:00,21.00,24.0,[100.0],[uV],"Movement time:1, 1:101, 2:33, ?:1, R:8, W:74"
34,SC4761,1989-11-13 16:35:00,21.92,24.0,[100.0],[uV],"1:47, 2:39, 3:12, ?:3, R:8, W:28"


Canales del PSG: ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']
Prefiltro de adquisición EEG: {'EEG Fpz-Cz': 'HP:0.5Hz LP:100Hz [enhanced cassette BW]', 'EEG Pz-Oz': 'HP:0.5Hz LP:100Hz [enhanced cassette BW]'}
Advertencias (36):
  - SC4041E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 77100 s; las épocas sin señal se descartan al segmentar
  - SC4042E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 83760 s; las épocas sin señal se descartan al segmentar
  - SC4051E0-PSG.edf: el hipnograma termina en 86400 s y la señal en 81660 s; las épocas sin señal se descartan al segmentar
  ... y 33 más


Observaciones:

- Todos los registros tienen los dos canales EEG a 100 Hz, en µV, con prefiltro de hardware 0.5–100 Hz.
- Los conteos de etiquetas son **segmentos** del hipnograma, no épocas (un segmento W puede durar horas).
- En todos los registros el hipnograma se extiende más allá del final de la señal: esas épocas se descartan y se
  contabilizan como faltantes (ver *Tratamiento de datos faltantes*).

### Carga optimizada

- **Elección de columnas:** cada EDF tiene 7 canales, y se cargan solo los 2 de EEG. Además de ahorrar memoria, evita
  un costo oculto: al leer canales con frecuencias distintas, MNE lleva todos a la frecuencia máxima.
- **Tipado:** las épocas se guardan en `float32`, porque 24 h de señal en `float64` no se justifican para datos de 12
  bits. Las features se calculan en `float64`, para no acumular error de redondeo.
- **Tabla tidy:** `sleep_stage` y `channel` se guardan como categóricas, y la tabla se escribe en Parquet, que conserva
  los tipos.

> **Prompt 7.** *Segmentá en épocas de 30 s y pasá las etapas a AASM, con S3 y S4 como N3. Las etapas M y ? no se imputan. Recortá la vigilia a 30 min antes y después del sueño nocturno para que no domine, y filtrá de 0,5 a 30 Hz sobre la señal continua.*

## Transform

Cada registro se procesa **secuencialmente**, para no tener varias señales en memoria:

1. **Épocas**: los segmentos del hipnograma se dividen en épocas de 30 s. Se verifica que las anotaciones sean
   contiguas y estén alineadas a 30 s; si hubiera huecos, el pipeline se detiene.
2. **Recorte al período de sueño nocturno**: los registros SC duran ~20 h y la mayor parte es vigilia diurna. Se
   conservan **30 min de W antes de la primera época de sueño y 30 min después de la última** (práctica habitual con este
   dataset). Sin recorte, W representaría ~70 % de las épocas y dominaría cualquier comparación.
   - La primera y la última época de sueño se buscan solo **dentro de la franja 21:00–12:00** (hora local del
     registro, `NIGHT_START_HOUR` / `NIGHT_END_HOUR`). Sin esta restricción, las **siestas diurnas** de los sujetos
     mayores (por ejemplo, a las 13:00 o a las 17:00) estiraban la "noche" a casi 24 h.
   - La franja se eligió después de comparar alternativas. Separar bloques de sueño por huecos de vigilia partía las
     noches con despertares largos, que son frecuentes en las personas mayores.
   - Las épocas recortadas no se eliminan de la tabla de épocas: quedan marcadas (`in_analysis_window = False`).
3. **Carga selectiva**: solo se leen los 2 canales EEG. Esto evita que MNE suba a 100 Hz los canales de 1 Hz y reduce
   la memoria.
4. **Filtro pasa-banda 0.5–30 Hz** (FIR de fase cero, el valor por defecto de MNE), aplicado a la señal **continua**
   recortada, con 30 s de margen a cada lado, antes de segmentar. Filtrar cada época por separado introduciría
   efectos de borde.
   - El pasa-altos de 0.5 Hz elimina la deriva lenta. El hardware ya aplica 0.5 Hz, pero así se garantiza el mismo
     corte en todos los registros.
   - El pasa-bajos de 30 Hz coincide con el límite superior de la banda beta analizada y atenúa actividad muscular y
     ruido de red (50 Hz, cerca de Nyquist).
   - No se aplica ninguna otra transformación (ni re-referencia, ni ICA, ni normalización): para las métricas
     elegidas no son necesarias y agregarían supuestos.
5. **Array de épocas** en µV, float32 (precisión suficiente para la señal y la mitad de memoria). Las features se
   calculan en float64, porque acumular sumas espectrales en float32 introduce errores de ~1e-6.
6. **Features y flags de artefacto**, guardados como Parquet por registro (caché).

El caché se invalida automáticamente si cambian los parámetros metodológicos de `config.py` o la versión del código
del Transform (se guarda un hash de ambos).

A continuación se ilustra el efecto del filtro sobre una época y luego se ejecuta el pipeline completo.

In [ ]:
import plotly.graph_objects as go
from src import plots

rec = files.iloc[0]
psg, hyp = config.PROJECT_ROOT / rec.PSG, config.PROJECT_ROOT / rec.Hypnogram
info = preprocessing.inspect_recording(psg, hyp)
ep = preprocessing.mark_analysis_window(
    preprocessing.annotations_to_epochs(preprocessing.read_hypnogram(hyp), info["psg_duration_s"]),
    info["start_datetime"])
example = ep[ep["in_analysis_window"] & ep["has_signal"] & (ep["stage"] == "N2")].iloc[20]

raw_unfiltered = preprocessing.load_eeg(psg, hyp)
raw_filtered = preprocessing.crop_and_filter(raw_unfiltered.copy(), ep)
x_raw = preprocessing.extract_epoch_array(raw_unfiltered, np.array([example.start_sec]))[0, 0]
x_filt = preprocessing.extract_epoch_array(raw_filtered, np.array([example.start_sec]))[0, 0]
del raw_unfiltered, raw_filtered

t = np.arange(x_raw.size) / config.EXPECTED_SFREQ_HZ
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=x_raw, name="Original", line=dict(color=plots.INK_MUTED, width=1)))
fig.add_trace(go.Scatter(x=t, y=x_filt, name="Filtrado 0.5–30 Hz", line=dict(color=plots.STAGE_COLORS["N2"], width=1)))
fig.update_xaxes(title="Tiempo dentro del epoch [s]")
fig.update_yaxes(title="Amplitud [µV]")
plots._style(fig, f"Fpz-Cz · {rec.recording_id} · epoch {example.epoch_number} (N2)", height=360).show()

In [ ]:
from src import pipeline

print("Versiones:", pipeline.library_versions())
results = pipeline.run_pipeline(manifest)
tidy, epochs, recordings, missing = results["tidy"], results["epochs"], results["recordings"], results["missing"]
print(f"Tabla tidy: {tidy.shape[0]:,} filas × {tidy.shape[1]} columnas, "
      f"{tidy.memory_usage(deep=True).sum() / 1e6:.1f} MB en memoria")

Versiones: {'python': '3.13.11', 'numpy': '2.2.6', 'pandas': '2.2.3', 'scipy': '1.15.3', 'mne': '1.9.0', 'mne-bids': '0.20.0', 'antropy': '0.1.9', 'pyarrow': '25.0.1', 'plotly': '7.1.0', 'streamlit': '1.65.0', 'scikit-learn': '1.6.1'}


INFO src.pipeline: Tidy: 83192 filas x 27 columnas (27.1 MB en memoria)


Tabla tidy: 83,192 filas × 27 columnas, 27.1 MB en memoria


## Hipnograma y épocas

Las etiquetas originales (R&K) se mapean a la nomenclatura AASM:

| Hipnograma | Etapa | Tratamiento |
|---|---|---|
| `Sleep stage W` | W | Puntuada |
| `Sleep stage 1` | N1 | Puntuada |
| `Sleep stage 2` | N2 | Puntuada |
| `Sleep stage 3`, `Sleep stage 4` | N3 | Puntuada (R&K S3 + S4 = AASM N3) |
| `Sleep stage R` | REM | Puntuada |
| `Movement time` | MOVE | **No es una etapa**: se conserva, no se imputa |
| `Sleep stage ?` | UNK | **Desconocida**: se conserva, no se imputa |

MOVE y `?` no se reasignan a una etapa vecina: hacerlo inventaría información que el experto no registró. Se
conservan con su propia categoría, figuran en las bases y se excluyen de las comparaciones entre etapas.

In [ ]:
display(recordings[["recording_id", "subject", "night", "age", "sex", "lights_off",
                    "n_epochs_hypnogram", "n_epochs_analyzed"]])
counts = (epochs.assign(grupo=np.select(
              [~epochs["has_signal"], ~epochs["in_analysis_window"]],
              ["sin señal", "recortado (vigilia)"], "analizado"))
          .pivot_table(index="stage", columns="grupo", values="epoch_number", aggfunc="count", fill_value=0))
display(counts)

,recording_id,subject,night,age,sex,lights_off,n_epochs_hypnogram,n_epochs_analyzed
0,SC4041,4,1,34,F,23:12:00,2880,1236
1,SC4042,4,2,34,F,23:35:00,2880,1204
2,SC4051,5,1,28,F,01:22:00,2880,672
3,SC4052,5,2,28,F,00:35:00,2880,1248
4,SC4061,6,1,31,F,00:16:00,2880,843
...,...,...,...,...,...,...,...,...
31,SC4652,65,2,88,F,23:00:00,2880,1363
32,SC4731,73,1,97,M,22:30:00,2880,1637
33,SC4732,73,2,97,M,22:00:00,2880,1747
34,SC4761,76,1,90,M,23:28:00,2880,1703


grupo,analizado,recortado (vigilia),sin señal
stage,,,
MOVE,39,0,0
N1,4030,319,0
N2,16641,375,0
N3,2610,24,0
REM,6855,0,0
UNK,160,0,5138
W,11261,56228,0


In [ ]:
obs_preview = tidy.rename(columns={"sleep_stage": "stage_code", "start_time_sec": "start_sec"})
obs_preview["channel"] = obs_preview["channel"].astype(str).str.replace("EEG ", "")
onset = (obs_preview[obs_preview["stage_code"].isin(["N1", "N2", "N3", "REM"])]
         .groupby("recording_id")["start_sec"].min())
obs_preview["hours_since_sleep_onset"] = (obs_preview["start_sec"] - obs_preview["recording_id"].map(onset)) / 3600
obs_preview["stage_code"] = obs_preview["stage_code"].astype(str)
plots.hypnogram(obs_preview, files.iloc[0].recording_id).show()

## Control de calidad

Además de las validaciones de entrada (sección *Exploración inicial*), sobre la tabla resultante se verifica que:

- no haya claves (registro, época, canal) duplicadas;
- cada época tenga todos los canales;
- todas las épocas empiecen en múltiplos de 30 s;
- las etapas sean válidas;
- la potencia relativa esté en [0, 1] y sume 1;
- `is_artifact` sea coherente con `artifact_reason`;
- no haya features NaN en filas sin artefacto.

In [ ]:
show_checks(validation.validate_tidy(tidy))

  ✔ Sin claves (recording, epoch, channel) duplicadas
  ✔ Canales: ['EEG Fpz-Cz', 'EEG Pz-Oz']
  ✔ Cada epoch tiene todos los canales
  ✔ Inicios de epoch múltiplos de 30 s
  ✔ Etapas válidas: ['MOVE', 'N1', 'N2', 'N3', 'REM', 'UNK', 'W']
  ✔ Potencia relativa en [0, 1] y suma 1 ± 1e-09 (sin artefactos)
  ✔ is_artifact coherente con artifact_reason
  ✔ Sin features NaN en filas no artefactadas


> **Prompt 8.** *Generá un perfil por columna con nulos, ceros y negativos, y explicame qué significa cada uno antes de decidir nada. Si algo no se puede calcular, dejalo nulo: no imputes.*

## Tratamiento de datos faltantes

Se distinguen cuatro situaciones. Ninguna se resuelve por imputación:

1. **Épocas del hipnograma sin señal**: el hipnograma continúa después del final del PSG. Sin señal no hay
   features: se excluyen y se cuentan.
2. **Vigilia fuera de la ventana de análisis**: no es un faltante, sino una **exclusión por diseño** (recorte de
   vigilia). Se reporta para que el volumen de datos descartado sea explícito.
3. **Etapa desconocida (`?`) o Movement time dentro de la ventana**: la señal existe y se calculan sus features, pero
   la etapa es desconocida. Se conservan como UNK/MOVE y se excluyen de las comparaciones entre etapas.
   Transformarlas en una etapa (por ejemplo, copiando la de la época anterior) sería fabricar una etiqueta.
4. **Features no calculables**: si la señal es plana o tiene valores no finitos, la Sample Entropy, la Spectral
   Entropy y las potencias relativas no están definidas. Se guardan como NaN (`NULL` en SQL) y no como 0, porque
   `antropy` devolvería 0, que se leería como "regularidad perfecta".

In [ ]:
display(missing.style.format({"percent": "{:.2f} %"}).hide(axis="index"))

level,item,count,denominator,percent,cause,decision
epoch,Epoch del hipnograma sin señal EEG,5138,103680,4.96 %,El hipnograma se extiende más allá del final del PSG,Excluido: no hay señal de la cual extraer features
epoch,Fuera de la ventana nocturna (vigilia y siestas),56946,103680,54.92 %,Registros de ~20 h; se conservan 30 min de W antes/después del sueño nocturno (21:00-12:00),Excluido por diseño (no es un faltante): evita que W y las siestas diurnas dominen el análisis
epoch,Etapa desconocida ('?') en ventana,160,103680,0.15 %,El experto no asignó etapa,Se conserva con etapa UNK; no se imputa; se excluye de comparaciones entre etapas
epoch,Movement time en ventana,39,103680,0.04 %,Movimiento corporal: el experto no pudo asignar etapa,Se conserva con etapa MOVE; no se imputa; se excluye de comparaciones entre etapas
epoch,Etapa sin mapeo,0,103680,0.00 %,Descripción del hipnograma no contemplada en STAGE_MAPPING,Debe ser 0 (la validación de cada registro lo impide)
feature,delta_abs = NaN,0,83192,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,theta_abs = NaN,0,83192,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,alpha_abs = NaN,0,83192,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,sigma_abs = NaN,0,83192,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto
feature,beta_abs = NaN,0,83192,0.00 %,"No calculable (señal plana o no finita, o SampEn sin coincidencias)",Se deja NaN (no se imputa); la fila conserva su flag de artefacto


### Perfil de nulos y ceros

Antes de decidir un tratamiento hay que distinguir **qué significa** cada nulo y cada cero. Un valor faltante no
siempre es un dato perdido, y un cero puede ser un valor válido, un faltante disfrazado o un sensor caído.

In [ ]:
from src import features, quality

feature_columns = features.FEATURE_COLUMNS
display(quality.data_profile(tidy[feature_columns]).round(3))
print("Nulos en columnas de texto de la tabla tidy:",
      tidy.select_dtypes(exclude="number").isna().sum().loc[lambda s: s > 0].to_dict())
print("Nulos en recordings:", recordings.isna().sum().loc[lambda s: s > 0].to_dict() or "ninguno")
print("Ceros en epochs:", quality.data_profile(epochs).query("ceros > 0")["ceros"].to_dict())

,nulos,infinitos,ceros,negativos,mínimo,mediana,máximo
delta_abs,0,0,0,0,0.794,52.460,5787.541
theta_abs,0,0,0,0,0.036,8.746,702.771
alpha_abs,0,0,0,0,0.007,3.808,191.058
sigma_abs,0,0,0,0,0.002,1.950,217.176
beta_abs,0,0,0,0,0.017,1.761,954.916
delta_rel,0,0,0,0,0.047,0.757,0.995
theta_rel,0,0,0,0,0.002,0.109,0.625
alpha_rel,0,0,0,0,0.001,0.047,0.823
sigma_rel,0,0,0,0,0.000,0.025,0.336
beta_rel,0,0,0,0,0.000,0.024,0.611


Nulos en columnas de texto de la tabla tidy: {'artifact_reason': 83019, 'outlier_features': 75047}
Nulos en recordings: ninguno
Ceros en epochs: {'epoch_number': 36, 'start_sec': 36}


In [ ]:
# Valores casi nulos: el cero exacto no aparece, pero sí valores muy bajos.
low_amplitude = tidy[tidy["rms"] < 2]
print(f"RMS < 2 µV: {len(low_amplitude):,} filas · por registro:",
      low_amplitude["recording_id"].value_counts().head(4).to_dict())
display(tidy.groupby(["recording_id", "channel"], observed=True)["rms"].median().unstack().round(1)
        .sort_values("EEG Pz-Oz").head(3))
columns = ["recording_id", "epoch_number", "channel", "sleep_stage", "sample_entropy", "rms", "peak_to_peak"]
display(tidy.loc[tidy["sample_entropy"] < 0.05, columns].round(3))

RMS < 2 µV: 1,223 filas · por registro: {'SC4221': 750, 'SC4762': 298, 'SC4761': 98, 'SC4501': 77}


channel,EEG Fpz-Cz,EEG Pz-Oz
recording_id,,
SC4221,15.5,1.9
SC4731,11.1,3.7
SC4732,11.6,4.0


,recording_id,epoch_number,channel,sleep_stage,sample_entropy,rms,peak_to_peak
76924,SC4761,528,EEG Fpz-Cz,W,0.037,20.299,247.366
76972,SC4761,552,EEG Fpz-Cz,W,0.037,21.428,259.533
80011,SC4761,2071,EEG Pz-Oz,W,0.045,29.599,255.272
80955,SC4762,1266,EEG Pz-Oz,UNK,0.046,26.158,190.774
82204,SC4762,1891,EEG Fpz-Cz,W,0.030,20.551,186.792
...,...,...,...,...,...,...,...
82726,SC4762,2152,EEG Fpz-Cz,UNK,0.027,18.871,152.698
82727,SC4762,2152,EEG Pz-Oz,UNK,0.021,23.128,243.287
83092,SC4762,2335,EEG Fpz-Cz,UNK,0.007,20.090,213.975
83093,SC4762,2335,EEG Pz-Oz,UNK,0.017,15.961,172.082


**Lectura de los perfiles** (valores de esta ejecución):

- **Nulos en features: 0.** La regla de "NaN si no es calculable" existe, pero no se activó en esta muestra.
- **Nulos estructurales:** `artifact_reason` es nulo en el 99.8 % de las filas y `outlier_features` en el 90 %. No son
  datos perdidos: significan "sin artefacto" y "sin outlier". **No se imputan.**
- **Ceros exactos: 0 en las 15 features.** Es lo esperable: una potencia o una entropía exactamente en 0 solo
  aparecería con señal plana. Los únicos ceros están en `epoch_number` y `start_sec` de los 36 registros, y son la
  primera época, un valor válido.
- **Consecuencia práctica:** a diferencia del dataset de clase, ninguna variable tiene Q1 = 0, así que el IQR no se
  anula. El problema de la clase con `RobustScaler` no aparece.
- **Ceros en la señal:** el cero exacto no sirve como criterio, porque la conversión digital del EDF tiene offset y
  ningún valor cae en 0 µV. Lo que importa es la señal **constante**, y la racha constante más larga es de 70 muestras
  (0.7 s), en un tramo saturado (ver la subsección *Saturación*).
- **Casi ceros:** el criterio `flat_signal` (SD < 0.5 µV) no se activó y el RMS mínimo es 0.99 µV. Las 1 223 filas con
  RMS < 2 µV se concentran en SC4221 (750): su canal Pz-Oz tiene una mediana de 1.9 µV, frente a 6.9 µV del conjunto.
  Es una **diferencia del registro**, no de épocas aisladas.
- **Sample Entropy casi nula:** 56 filas < 0.05, todas en SC4761/SC4762. 50 son épocas `?` de SC4762, que el experto
  no pudo puntuar, y 6 son W. Tienen RMS de 10–42 µV y pico a pico de 127–260 µV, así que no son señal plana. Es
  compatible con un transitorio breve: infla el desvío estándar, y con él la tolerancia `r = 0.2·SD`, de modo que casi
  todos los patrones coinciden. Son valores **extremos de cálculo**, no evidencia de "regularidad". Las épocas `?` ya
  quedan fuera de las comparaciones entre etapas.

> **Prompt 9.** *Aplicá los métodos de outliers de la clase 5: z-score, Tukey fences, k-means, k-NN, LOF e Isolation Forest, todos marcando la misma proporción. Decime si coinciden y cuánto cambian las medianas por etapa si excluyo cada grupo.*

> **Prompt 10.** *Reemplazá el umbral de 500 µV por un criterio de saturación, porque el rango físico del EDF no permite alcanzarlo. Y calculá Tukey dentro de cada noche, para marcar épocas raras y no personas distintas.*

## Tratamiento de outliers y artefactos

### Outlier estadístico ≠ artefacto de señal

| | Outlier estadístico | Artefacto de señal |
|---|---|---|
| Definición | Valor extremo respecto de la distribución observada | El registro no refleja actividad cerebral |
| Criterio | Regla de Tukey: fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR] | Criterios físicos sobre la señal |
| Ejemplo | Una onda lenta de gran amplitud en N3 | Electrodo desconectado, saturación, movimiento |
| Decisión | **Solo se marca** (`is_stat_outlier`, `outlier_features`) | **Se marca** (`is_artifact`, `artifact_reason`) y se excluye por defecto de los análisis |

**Ninguna fila se elimina.** Un valor extremo puede ser fisiología real, y eliminarlo sesgaría justamente las etapas
más interesantes.

### Criterios de artefacto (configurables en `config.py`)

- `non_finite`: algún valor NaN/inf en la época.
- `flat_signal`: desvío estándar < `ARTIFACT_MIN_STD_UV` (0.5 µV), compatible con un electrodo desconectado.
- `clipped`: al menos `ARTIFACT_MIN_CLIPPED_SAMPLES` muestras (10, es decir 0.1 s) en el límite físico del EDF, contadas
  sobre la señal sin filtrar. Indica que el conversor se saturó y que la señal real quedó recortada. Un umbral de
  amplitud no sirve aquí (ver *Saturación*): las ondas lentas grandes de N3 en Fpz-Cz llegan a ≈ 437 µV tras el filtro, y
  un umbral más bajo marcaría sueño profundo como artefacto.

### Outliers estadísticos

La regla de Tukey se aplica **por registro × canal × etapa**. Las distribuciones difieren legítimamente entre etapas
(N3 tiene más delta que REM por definición) y también entre noches, por edad o ganancia del equipo: cada época se
compara con su propia noche (ver *¿Épocas raras o registros distintos?*). Las potencias absolutas se evalúan en log10,
porque su distribución es muy asimétrica.

In [ ]:
artifact_summary = (tidy.groupby(["recording_id", "channel"], observed=True)
                    .agg(observaciones=("is_artifact", "size"), artefactos=("is_artifact", "sum"),
                         outliers_estadisticos=("is_stat_outlier", "sum")))
artifact_summary["% outliers"] = (100 * artifact_summary["outliers_estadisticos"] / artifact_summary["observaciones"]).round(1)
display(artifact_summary)
print("Motivos de artefacto:", tidy["artifact_reason"].value_counts().to_dict() or "ninguno")
print("Features más frecuentes entre los outliers estadísticos:")
display(tidy["outlier_features"].str.split(";").explode().value_counts().to_frame("filas"))

observaciones  artefactos  outliers_estadisticos  % outliers
recording_id channel                                                                 
SC4041       EEG Fpz-Cz           1236           0                    110         8.9
             EEG Pz-Oz            1236           0                    156        12.6
SC4042       EEG Fpz-Cz           1204           0                     92         7.6
             EEG Pz-Oz            1204           1                    156        13.0
SC4051       EEG Fpz-Cz            672           2                     57         8.5
...                                ...         ...                    ...         ...
SC4732       EEG Pz-Oz            1747           0                    164         9.4
SC4761       EEG Fpz-Cz           1703           0                    175        10.3
             EEG Pz-Oz            1703           0                    228        13.4
SC4762       EEG Fpz-Cz           1476           0                    262        17.8
             EEG Pz-Oz            1476           0                    123         8.3

[72 rows x 4 columns]

Motivos de artefacto: {'clipped': 173}
Features más frecuentes entre los outliers estadísticos:


,filas
outlier_features,
alpha_rel,2993
rms,2870
peak_to_peak,2820
sample_entropy,1968
spectral_entropy,1921
total_power,1804
delta_rel,1743
delta_abs,1666


La figura siguiente responde: **¿dónde caen los valores extremos de amplitud?** Los puntos son los valores fuera de
los bigotes del box plot. Las amplitudes más altas aparecen en N3 y corresponden a ondas lentas, no a artefactos.

In [ ]:
plots.amplitude_by_stage(obs_preview).show()

### Diagnóstico de outliers

Esta subsección aplica los métodos de la clase 05 y los adapta al EEG. Las preguntas son qué tan válidas son las
reglas usadas, si los métodos coinciden entre sí y cuánto cambiarían los resultados según cómo se traten los valores
extremos. Este análisis justificó dos decisiones del pipeline: marcar la **saturación** como artefacto y calcular
Tukey **dentro de cada noche**. Los detectores multivariados son exploratorios y no modifican ninguna tabla.

#### Saturación: por qué reemplazó al umbral de amplitud

Los EDF tienen un rango físico fijo por canal y registro (entre ±147 y ±248 µV en esta muestra). Una señal que supera
ese rango queda **recortada** en el límite. Por eso se cuentan las muestras en el límite físico, sobre la señal **sin
filtrar**, porque el filtro redondea las mesetas.

In [ ]:
from src import outliers

is_clipped = tidy["artifact_reason"].eq("clipped")
print(f"Pico a pico máximo tras el filtro: {tidy['peak_to_peak'].max():.0f} µV")
print(f"Filas saturadas: {is_clipped.sum()} ({100 * is_clipped.mean():.2f} %)")
display(pd.crosstab(tidy["sleep_stage"].astype(str), tidy["channel"].astype(str),
                    values=is_clipped, aggfunc="sum").fillna(0).astype(int))
print("Por registro:", tidy.loc[is_clipped, "recording_id"].value_counts().head(5).to_dict())

Pico a pico máximo tras el filtro: 437 µV
Filas saturadas: 173 (0.21 %)


channel,EEG Fpz-Cz,EEG Pz-Oz
sleep_stage,,
MOVE,0,6
N1,1,6
N2,6,22
N3,0,2
REM,6,9
UNK,0,0
W,87,28


Por registro: {'SC4201': 36, 'SC4162': 35, 'SC4061': 24, 'SC4161': 19, 'SC4651': 17}


**Corrección del criterio.** Una versión anterior del pipeline usaba un umbral de pico a pico de 500 µV y daba "0
artefactos". Ese resultado no demostraba que los datos estuvieran limpios: el rango físico llega como máximo a −248 µV y
+212 µV, así que el pico a pico sin filtrar no puede superar 460 µV, y el máximo observado es de 420 µV. La señal se recorta antes de
alcanzar el umbral, que era **prácticamente inalcanzable**. Por eso se reemplazó por el criterio de saturación.

- Hay **173 filas (0.21 %)** con al menos 10 muestras saturadas (0.1 s).
- El 66 % está en W (115), lo que es compatible con movimientos.
- **Solo 2 de las 5 220 filas de N3 (0.04 %)**, ambas en SC4162. Son ondas lentas grandes con el pico recortado: la
  actividad es real, pero la señal quedó distorsionada. El criterio prácticamente no afecta al sueño profundo.
- Estas filas conservan sus features y se excluyen por defecto de los análisis, igual que cualquier artefacto.

#### Reglas univariadas: ¿qué tan válido es el umbral?

Como se planteó en clase, `|z| > 3` supone normalidad. La tabla muestra la forma de cada distribución dentro de canal
× etapa (sin artefactos) y el porcentaje que marca cada regla. Bajo normalidad, `|z| > 3` marcaría 0.27 %.

In [ ]:
display(outliers.compare_univariate_rules(tidy).round(2))

,asimetría,curtosis,A²,% |z| > 3,% Tukey 1.5,% Tukey 3
variable,,,,,,
delta_abs,-0.12,1.17,4.09,1.06,2.53,0.15
total_power,-0.42,1.66,8.65,1.46,2.77,0.23
delta_rel,-0.58,0.10,34.09,0.36,0.91,0.03
alpha_rel,1.89,5.70,192.94,1.74,6.40,1.64
sample_entropy,0.35,1.33,6.87,0.72,1.56,0.17
spectral_entropy,-0.37,0.19,8.44,0.42,1.21,0.05
rms,1.84,8.34,94.83,1.42,3.39,0.95
peak_to_peak,1.69,6.40,113.61,1.46,3.81,1.04


- **Ninguna variable es normal** (A² entre 4.1 y 193, frente a un valor crítico de ≈ 1.1 al 1 %). Además, las épocas de
  una noche no son independientes, así que A² se usa como descripción y no como prueba formal.
- Las colas pesadas (`rms` con curtosis 8.3, `peak_to_peak` con 6.4) llevan a `|z| > 3` a marcar hasta 1.7 %, seis
  veces lo esperado. El umbral de 3 desvíos **no es un percentil real** con estos datos.
- Tukey no asume normalidad, por eso es la regla más razonable aquí. Pero 1.5·IQR marca hasta 6.4 %, y sobre 8
  variables se acumula.

#### ¿Épocas raras o registros distintos?

Una versión anterior comparaba cada época contra **todos los registros** de su canal y etapa, y marcaba 14.3 % de las
filas. La figura compara ese criterio con el actual, que usa como referencia **la misma noche**.

In [ ]:
tukey_between, tukey_rates = outliers.tukey_rate_by_recording(tidy)
print(f"Filas marcadas: entre registros {100 * tukey_between.mean():.1f} % · "
      f"dentro del registro {100 * tidy['is_stat_outlier'].mean():.1f} %")
plots.outlier_rate_by_recording(tukey_rates).show()

Filas marcadas: entre registros 14.3 % · dentro del registro 9.8 %


Con la referencia global, SC4502, SC4501 y SC4221 tienen 42 % de "outliers", mientras que SC4102 tiene 1 %. Esa
regla detecta **qué persona o qué ganancia tiene el registro**, no épocas extrañas. Dentro de cada noche la tasa queda
entre 7 % y 14 % en todos los registros, con 9.8 % en total. El grupo de referencia cambia más el resultado que el
umbral, y por eso el pipeline usa la referencia dentro de la noche.

#### Detectores multivariados

Una fila puede ser normal en cada variable y rara en la combinación. Se aplican 5 detectores de la clase sobre 8
features (log de potencias y amplitud, y entropías), con z robusto por registro × canal, de modo que cada época se mida
contra su noche. Siguiendo la conclusión de la clase, todos marcan **la misma proporción de filas (1 %)**, y DBSCAN
queda afuera porque no da un puntaje continuo.

No hay una etiqueta de anomalía como `NSP`. Como referencias parciales se usan la **saturación** y las épocas
**Movement time** del experto.

In [ ]:
scores = outliers.multivariate_scores(tidy)
flags = outliers.flag_top(scores)
votes = flags.sum(axis=1)
consensus = votes >= config.OUTLIER_MIN_VOTES

print("Índice de Jaccard entre las filas marcadas por cada detector:")
display(outliers.overlap(flags).round(2))
print("ROC-AUC contra las referencias parciales:")
display(outliers.reference_auc(scores, {"Saturación": is_clipped, "Movement time": tidy["sleep_stage"].eq("MOVE")}).round(3))
print("Votos por fila:", votes.value_counts().sort_index().to_dict())
print(f"Consenso (≥ {config.OUTLIER_MIN_VOTES} de 5): {consensus.sum()} filas ·",
      tidy.loc[consensus, "sleep_stage"].astype(str).value_counts().to_dict())

Índice de Jaccard entre las filas marcadas por cada detector:


,Distancia al centro,k-means,k-NN,LOF,Isolation Forest
Distancia al centro,1.00,0.55,0.23,0.03,0.64
k-means,0.55,1.00,0.30,0.03,0.51
k-NN,0.23,0.30,1.00,0.09,0.21
LOF,0.03,0.03,0.09,1.00,0.03
Isolation Forest,0.64,0.51,0.21,0.03,1.00


ROC-AUC contra las referencias parciales:


,Saturación,Movement time
Distancia al centro,0.960,0.714
k-means,0.909,0.717
k-NN,0.935,0.719
LOF,0.506,0.455
Isolation Forest,0.955,0.711


Votos por fila: {0: 80925, 1: 1268, 2: 366, 3: 404, 4: 197, 5: 32}
Consenso (≥ 4 de 5): 229 filas · {'W': 177, 'N1': 23, 'N3': 11, 'N2': 8, 'REM': 4, 'UNK': 4, 'MOVE': 2}


- **Los detectores no coinciden mucho.** La superposición va de 0.03 a 0.64. Distancia al centro e Isolation Forest
  miden casi lo mismo (extremos globales) y LOF encuentra otra cosa (rarezas locales). No existe "el" conjunto de
  outliers: depende de la definición.
- **Saturación:** ROC-AUC de 0.91–0.96, salvo LOF (0.51). El resultado es en parte esperable, porque la amplitud está
  entre las features. Pero marcar el 1 % es exigente: solo 10 de las 173 filas saturadas llegan al consenso. Para
  encontrar saturación conviene el criterio físico directo, no un detector.
- **Movement time:** ROC-AUC de 0.71–0.72, salvo LOF (0.46), con solo 78 filas de referencia. Los detectores ordenan
  moderadamente bien, pero **no** reemplazan al experto para identificar movimiento.
- **Consenso:** 229 filas (0.28 %), 177 de ellas en W y 61 en SC4611. Sin una etiqueta no se puede afirmar que sean
  artefactos.

#### ¿Cuánto cambian los resultados?

Esta es la pregunta que decide el tratamiento. Para cada criterio se calcula el mayor cambio relativo de las medianas
por etapa y canal si se excluyeran esas filas.

In [ ]:
sensitivity = outliers.exclusion_sensitivity(tidy, {
    "Tukey entre registros": tukey_between,
    "Tukey dentro del registro": tidy["is_stat_outlier"],
    "Saturación": is_clipped,
    "Consenso multivariado": consensus,
})
display(sensitivity.round(2))

,filas excluidas,% excluido,Δ% sample_entropy,Δ% spectral_entropy,Δ% delta_rel,Δ% alpha_rel
criterio,,,,,,
Tukey entre registros,11883,14.35,7.39,3.00,3.30,16.98
Tukey dentro del registro,8101,9.78,2.23,1.42,6.12,6.82
Saturación,167,0.20,0.43,0.18,0.15,1.09
Consenso multivariado,223,0.27,0.10,0.14,0.15,0.57


Excluir por Tukey quitaría 10–14 % de los datos y movería `alpha_rel` hasta 17 %. Esto confirma que eliminarlos
**cambiaría los resultados**, sin ninguna evidencia de que esas filas estén mal. Excluir por saturación o por
consenso quita menos de 0.3 % de las filas y mueve las medianas menos de 1.1 %.

#### Resumen: cómo se trata cada caso

| Caso | Situación en estos datos | Tratamiento actual | Fundamento |
|---|---|---|---|
| Nulos en features | 0 | NaN sin imputar | Una feature no calculable no se inventa |
| Nulos en flags | Estructurales | Se conservan | Significan "sin marca" |
| Etapas `?` y `M` | 199 épocas | Conservadas como UNK/MOVE, sin imputar | Imputar sería fabricar una etiqueta |
| Ceros exactos | Solo índices válidos | Sin tratamiento | No representan faltantes |
| Amplitud baja | Concentrada en SC4221 Pz-Oz | Se conserva, documentada | Usar medidas relativas o normalizar por registro |
| Saturación | 173 filas (0.21 %) | Artefacto `clipped`: se conserva y se excluye por defecto | Criterio físico directo; casi no afecta a N3 (2 filas) |
| Outliers univariados | 9.8 % | Tukey dentro de cada noche: se marcan, nunca se eliminan | Excluirlos cambia los resultados sin justificación |
| Outliers multivariados | 229 filas de consenso | Solo diagnóstico | Sin etiqueta, no se pueden confirmar |

Las medianas por etapa, usadas en todo el análisis, son robustas: por eso el resultado casi no depende de estas filas.

> **Prompt 11.** *Por época y canal calculá potencia absoluta y relativa por banda con Welch, Sample Entropy, Spectral Entropy, RMS y pico a pico. Usá librerías mantenidas y dejá el resultado como tidy data en Parquet.*

## Extracción de features

Para cada época de 30 s (3000 muestras a 100 Hz) y cada canal:

| Feature | Definición | Unidad | Implementación |
|---|---|---|---|
| PSD | Welch: ventana Hann de 4 s (400 muestras), 50 % de solapamiento, detrend constante. Resolución 0.25 Hz | µV²/Hz | `scipy.signal.welch` |
| Potencia absoluta de banda | $P_b = \sum_{f_1 \le f < f_2} \mathrm{PSD}(f)\,\Delta f$ | µV² | Propia (suma de la PSD) |
| Bandas | delta 0.5–4, theta 4–8, alfa 8–12, sigma 12–16, beta 16–30 Hz | Hz | `config.FREQ_BANDS` |
| Potencia total | $P_{tot} = \sum_{0.5 \le f < 30} \mathrm{PSD}(f)\,\Delta f$ | µV² | Propia |
| Potencia relativa | $P_b / P_{tot}$ | — | Propia |
| Spectral Entropy | $-\sum_f p_f \log_2 p_f / \log_2 N_f$, con $p_f$ = PSD de Welch normalizada | [0, 1] | `antropy.spectral_entropy` |
| Sample Entropy | $-\ln(A/B)$; $B$ = pares de subsecuencias de largo $m=2$ con distancia de Chebyshev ≤ $r = 0.2\,\mathrm{SD}$; $A$ = ídem con $m+1$ | — | `antropy.sample_entropy` |
| RMS | $\sqrt{\overline{x^2}}$ | µV | NumPy |
| Pico a pico | $\max x - \min x$ | µV | NumPy |

Decisiones:

- **Intervalos semiabiertos y regla del rectángulo**: las bandas son disjuntas, suman exactamente la potencia total y
  la potencia total es consistente con la varianza (Parseval). Integrar con Simpson (como hace YASA) da una delta
  absoluta ~10 % menor, porque pondera menos el extremo de 0.5 Hz, donde la PSD es máxima. Por eso los valores
  **absolutos** dependen de la convención de integración y no deben compararse con trabajos que usen otra; las
  **relativas** no tienen este problema.
- **Las entropías no se implementan a mano**: se usa `antropy`, una librería mantenida y validada.
- **Spectral Entropy** se calcula sobre la PSD completa (0–50 Hz) de la señal ya filtrada: `antropy` no admite
  restringir el rango, y por encima de 30 Hz la potencia filtrada es despreciable.
- **Potencia absoluta y relativa no son intercambiables**: la absoluta depende de la amplitud global (que varía entre
  sujetos y canales); la relativa describe la *forma* del espectro.
- **Sample Entropy y Spectral Entropy miden cosas distintas**: la primera mide la predecibilidad temporal de la
  forma de onda; la segunda, cuán repartida está la potencia en frecuencia.

In [ ]:
feature_cols = features.FEATURE_COLUMNS
display(tidy[feature_cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
delta_abs,83192.0,141.053,262.490,0.794,21.328,52.460,142.183,5787.541
theta_abs,83192.0,14.033,19.529,0.036,4.816,8.746,16.017,702.771
alpha_abs,83192.0,7.029,10.219,0.007,2.017,3.808,7.859,191.058
sigma_abs,83192.0,4.038,7.429,0.002,0.915,1.950,4.444,217.176
beta_abs,83192.0,7.912,34.280,0.017,0.780,1.761,4.422,954.916
delta_rel,83192.0,0.717,0.185,0.047,0.625,0.757,0.857,0.995
theta_rel,83192.0,0.122,0.071,0.002,0.070,0.109,0.156,0.625
alpha_rel,83192.0,0.075,0.093,0.001,0.025,0.047,0.084,0.823
sigma_rel,83192.0,0.034,0.030,0.000,0.013,0.025,0.047,0.336
beta_rel,83192.0,0.052,0.075,0.000,0.011,0.024,0.058,0.611


## Tidy Data

La tabla procesada cumple los principios de *tidy data*:

- cada **fila** es una observación: **una época de 30 s de un canal EEG**;
- cada **columna** es una variable: identificadores, contexto, features y flags;
- cada **tabla** es un tipo de unidad observacional. Las épocas completas del hipnograma y los registros van en
  tablas aparte (`epochs.parquet`, `recordings.parquet`).

Se guarda en **Parquet** (`data/processed/features_tidy.parquet`): es un formato columnar, comprimido, que conserva
los tipos (categorías, booleanos, float64) y se lee mucho más rápido que un CSV.

In [ ]:
display(tidy.head())
print(tidy.dtypes.to_string())
print(f"\nMemoria: {tidy.memory_usage(deep=True).sum() / 1e6:.1f} MB · "
      f"Parquet: {pipeline.TIDY_PATH.stat().st_size / 1e6:.1f} MB")

,subject_id,recording_id,night,epoch_number,start_time_sec,sleep_stage,stage_original,channel,delta_abs,theta_abs,alpha_abs,sigma_abs,beta_abs,delta_rel,theta_rel,alpha_rel,sigma_rel,beta_rel,total_power,sample_entropy,spectral_entropy,rms,peak_to_peak,is_artifact,artifact_reason,is_stat_outlier,outlier_features
0,4,SC4041,1,752,22560.0,W,Sleep stage W,EEG Fpz-Cz,160.041275,22.947237,7.292657,5.880720,13.561372,0.763107,0.109417,0.034773,0.028040,0.064663,209.723261,1.135260,0.644313,14.792988,115.368408,False,None,False,None
1,4,SC4041,1,752,22560.0,W,Sleep stage W,EEG Pz-Oz,7.203409,6.687294,2.611215,1.119863,2.541609,0.357252,0.331655,0.129503,0.055539,0.126051,20.163390,1.618146,0.812323,4.558993,37.818740,False,None,False,None
2,4,SC4041,1,753,22590.0,W,Sleep stage W,EEG Fpz-Cz,209.694342,44.913856,7.225909,4.489356,12.433262,0.752249,0.161122,0.025922,0.016105,0.044603,278.756726,0.892456,0.620216,18.472021,182.236946,False,None,False,None
3,4,SC4041,1,753,22590.0,W,Sleep stage W,EEG Pz-Oz,10.877238,6.508806,2.694100,1.223040,2.840877,0.450514,0.269582,0.111584,0.050656,0.117664,24.144062,1.416990,0.781224,5.283261,48.111174,False,None,False,None
4,4,SC4041,1,754,22620.0,W,Sleep stage W,EEG Fpz-Cz,511.122568,100.012155,12.356350,5.853526,14.623671,0.793708,0.155306,0.019188,0.009090,0.022709,643.968269,0.790206,0.629726,28.498752,213.798080,False,None,False,None


subject_id             int64
recording_id          object
night                  int64
epoch_number           int64
start_time_sec       float64
sleep_stage         category
stage_original        object
channel             category
delta_abs            float64
theta_abs            float64
alpha_abs            float64
sigma_abs            float64
beta_abs             float64
delta_rel            float64
theta_rel            float64
alpha_rel            float64
sigma_rel            float64
beta_rel             float64
total_power          float64
sample_entropy       float64
spectral_entropy     float64
rms                  float64
peak_to_peak         float64
is_artifact             bool
artifact_reason       object
is_stat_outlier         bool
outlier_features      object

Memoria: 27.1 MB · Parquet: 12.6 MB


> **Prompt 12.** *Cargá los datos en una base transaccional SQLite en 3FN, con PK, FK y restricciones. Justificá 1FN, 2FN y 3FN con ejemplos del esquema.*

## SQLite OLTP

`data/database/sleep_oltp.sqlite` se reconstruye completo en cada ejecución: es un artefacto derivado de
`data/processed/`, y reconstruirlo garantiza que no queden filas obsoletas.

```
subject 1──N recording 1──N epoch N──1 hypnogram_label N──1 sleep_stage
                              │
                              1──N eeg_feature N──1 channel
```

Restricciones definidas:

- **PK** en todas las tablas; `eeg_feature` usa la clave compuesta (epoch_id, channel_id).
- **FK** activadas con `PRAGMA foreign_keys = ON`, que en SQLite están **desactivadas por defecto**.
- **UNIQUE** en (subject_id, night), (recording_id, epoch_number), nombres de canal, códigos de etapa y archivos.
- **NOT NULL** en todo identificador y atributo obligatorio. Las features admiten NULL a propósito, porque una
  feature no calculable es un faltante legítimo.
- **CHECK** de dominios: sexo ∈ {F, M}, potencias ≥ 0, Spectral Entropy ∈ [0, 1], y coherencia entre
  `is_artifact` y `artifact_reason`.
- **Índices** sobre las FK más consultadas.

In [ ]:
from src import database_oltp

database_oltp.build_oltp(tidy, recordings)
display(database_oltp.table_counts().to_frame())
show_checks(validation.validate_oltp(tidy))

INFO src.database_oltp: OLTP creada: data/database/sleep_oltp.sqlite


,rows
subject,18
recording,36
channel,2
sleep_stage,7
hypnogram_label,8
epoch,41596
eeg_feature,83192


  ✔ PRAGMA foreign_keys = ON
  ✔ PRAGMA foreign_key_check sin violaciones
  ✔ recording: 36 filas
  ✔ epoch: 41596 filas = epochs de la tidy
  ✔ eeg_feature: 83192 filas = filas de la tidy
  ✔ Potencia relativa recalculada en la vista coincide (max |dif| = 2.1e-15)


In [ ]:
# Demostración: las restricciones se aplican (cada intento se revierte).
import sqlite3
with database_oltp.connect() as conn:
    for description, sql in [
        ("epoch de un registro inexistente", "INSERT INTO epoch (recording_id, epoch_number, label_id) VALUES ('SC9999', 0, 1)"),
        ("(epoch, canal) duplicado", "INSERT INTO eeg_feature (epoch_id, channel_id, is_artifact) VALUES (1, 1, 0)"),
        ("artefacto sin motivo", "INSERT INTO eeg_feature (epoch_id, channel_id, is_artifact) VALUES (1, 2, 1)"),
    ]:
        try:
            conn.execute(sql)
            print(f"  ✘ {description}: NO fue rechazado")
        except sqlite3.IntegrityError as error:
            print(f"  ✔ {description}: rechazado ({error})")
    conn.rollback()
conn.close()

  ✔ epoch de un registro inexistente: rechazado (FOREIGN KEY constraint failed)
  ✔ (epoch, canal) duplicado: rechazado (UNIQUE constraint failed: eeg_feature.epoch_id, eeg_feature.channel_id)
  ✔ artefacto sin motivo: rechazado (CHECK constraint failed: (is_artifact = 0 AND artifact_reason IS NULL) OR (is_artifact = 1 AND artifact_reason IS NOT NULL))


## Normalización

**1NF.** Todos los atributos son atómicos (un valor por celda; no hay listas ni grupos repetidos) y cada tabla tiene
clave primaria. Las bandas de frecuencia se modelaron como columnas fijas (`delta_abs`, …) y no como un grupo
repetido: son un conjunto cerrado y conocido de variables de la misma observación.

**2NF.** No hay dependencias parciales de una clave compuesta:

- en `eeg_feature` (PK = epoch_id + channel_id), cada feature depende del par completo: la potencia delta de la época
  100 en Fpz-Cz no está determinada solo por la época ni solo por el canal;
- `start_sec = epoch_number × 30` dependería solo de `epoch_number`, que es parte de la clave candidata
  (recording_id, epoch_number). Por eso **no se almacena** y se calcula en la vista `v_epoch`;
- el flag de artefacto depende del canal (un parpadeo afecta a Fpz-Cz y no a Pz-Oz), por eso vive en `eeg_feature`
  y no en `epoch`.

**3NF.** No hay dependencias transitivas entre atributos no clave:

- la etiqueta original del hipnograma determina la etapa ("Sleep stage 4" → N3). Guardar ambas en `epoch` crearía
  `epoch → etiqueta → etapa`; por eso existe `hypnogram_label`, y `epoch` solo referencia la etiqueta;
- edad y sexo dependen del sujeto, no del registro: están en `subject`;
- la potencia total y las relativas se derivan de las absolutas (total = Σ bandas; relativa = banda / total).
  Almacenarlas sería redundante; las calcula la vista `v_eeg_feature`. La validación anterior confirma que la vista
  reproduce los valores de la tabla tidy (diferencia ~1e-15);
- los outliers estadísticos dependen de la distribución de cada noche, no de la fila: no son atributos de la
  observación y se calculan en el ETL hacia el Data Warehouse.

> **Prompt 13.** *Armá el almacén SQLite en esquema estrella, cargado desde la base transaccional. El grano es una época por canal y registro, con 3 dimensiones: etapa, registro y canal. El sujeto y el grupo etario van como jerarquía dentro de registro.*

## Data Warehouse

```
SQLite OLTP (3NF)
    ↓  Extract: una consulta SQL con JOIN sobre las vistas del OLTP
    ↓  Transform: dimensiones con surrogate keys, outliers estadísticos, tiempo desde el inicio del sueño
    ↓  Load: dimensiones primero, luego hechos (orden requerido por las FK)
SQLite Data Warehouse (estrella)
```

El ETL lee **del OLTP**, no de los Parquet: el OLTP es la fuente de verdad transaccional y el DW, su modelo analítico
derivado.

In [ ]:
from src import database_dw

dw_counts = database_dw.build_dw()
display(pd.Series(dw_counts, name="filas").to_frame())
show_checks(validation.validate_dw())

INFO src.database_dw: DW creado: data/database/sleep_dw.sqlite


,filas
dim_recording,36
dim_channel,2
dim_sleep_stage,8
fact_eeg_epoch,83192


  ✔ DW: PRAGMA foreign_key_check sin violaciones
  ✔ fact_eeg_epoch: 83192 filas = eeg_feature del OLTP
  ✔ Grano respetado: (registro, epoch, canal) único en la tabla de hechos
  ✔ Jerarquía de dim_recording coherente: cada sujeto tiene una edad, un sexo y un grupo
  ✔ 13 consultas analíticas reproducibles


## Esquema estrella

**Grano de `fact_eeg_epoch`:** *una observación de features EEG correspondiente a una época de 30 segundos, un canal,
un sujeto y un registro.*

```
   dim_recording ── fact_eeg_epoch ── dim_channel
                         │
                  dim_sleep_stage
```

| Tabla | Contenido |
|---|---|
| `fact_eeg_epoch` | Claves de las 3 dimensiones, `epoch_number` (dimensión degenerada), `start_sec`, `hours_since_sleep_onset`, las 15 métricas y los flags de calidad |
| `dim_recording` | Jerarquía grupo etario → sujeto → registro: grupo etario, sujeto, edad, sexo, noche, inicio, apagado de luces, duración total y analizada |
| `dim_channel` | Nombre, nombre corto, ubicación |
| `dim_sleep_stage` | Grano = etiqueta original del hipnograma, con su código AASM (S3 y S4 → N3): la jerarquía queda desnormalizada en la dimensión |

Decisiones:

- **Surrogate keys** enteras y asignadas en un orden determinístico, para que el DW sea idéntico en cada ejecución.
- **Las dimensiones no se referencian entre sí** (eso sería un copo de nieve): la tabla de hechos tiene las 3 claves.
- **El sujeto no es una dimensión aparte.** Cada registro pertenece a un único sujeto, así que sujeto y grupo etario
  forman una jerarquía dentro de `dim_recording`. Esa desnormalización es propia del esquema estrella, y deja las 3
  dimensiones del cubo. Los grupos etarios son <40, 40-64 y 65+.
- **Redundancia deliberada**: a diferencia del OLTP, se materializan las potencias total y relativas, el tiempo desde
  el inicio del sueño y los flags de outlier. Así las consultas analíticas necesitan un solo JOIN por dimensión.
- `hours_since_sleep_onset` permite comparar noches que empiezan a horas distintas.

In [ ]:
with database_dw.connect() as conn:
    for table in ["dim_recording", "dim_channel", "dim_sleep_stage"]:
        display(Markdown(f"**{table}**"))
        display(pd.read_sql(f"SELECT * FROM {table}", conn))
conn.close()

**dim_recording**

,recording_key,recording_id,age_group,subject_id,age,sex,night,start_datetime,lights_off,sampling_frequency_hz,psg_duration_h,analyzed_duration_h
0,1,SC4041,<40,4,34,F,1,1989-04-24T16:50:00,23:12:00,100.0,21.416667,10.300000
1,2,SC4042,<40,4,34,F,2,1989-04-25T15:04:00,23:35:00,100.0,23.266667,10.033333
2,3,SC4051,<40,5,28,F,1,1989-04-26T16:25:00,01:22:00,100.0,22.683333,5.600000
3,4,SC4052,<40,5,28,F,2,1989-04-27T15:42:00,00:35:00,100.0,23.383333,10.400000
4,5,SC4061,<40,6,31,F,1,1989-05-08T16:35:00,00:16:00,100.0,23.083333,7.025000
...,...,...,...,...,...,...,...,...,...,...,...,...
31,32,SC4652,65+,65,88,F,2,1989-11-07T14:40:00,23:00:00,100.0,23.666667,11.358333
32,33,SC4731,65+,73,97,M,1,1989-10-11T14:55:00,22:30:00,100.0,23.166667,13.641667
33,34,SC4732,65+,73,97,M,2,1989-10-12T14:30:00,22:00:00,100.0,21.000000,14.558333
34,35,SC4761,65+,76,90,M,1,1989-11-13T16:35:00,23:28:00,100.0,21.916667,14.191667


**dim_channel**

,channel_key,channel_name,short_name,location
0,1,EEG Fpz-Cz,Fpz-Cz,Bipolar frontal-central (Fpz referenced to Cz)
1,2,EEG Pz-Oz,Pz-Oz,Bipolar parietal-occipital (Pz referenced to Oz)


**dim_sleep_stage**

,stage_key,original_description,stage_code,stage_name,is_scored,stage_order
0,1,Sleep stage W,W,Wake,1,1
1,2,Sleep stage 1,N1,NREM 1,1,2
2,3,Sleep stage 2,N2,NREM 2,1,3
3,4,Sleep stage 3,N3,NREM 3 (slow-wave sleep),1,4
4,5,Sleep stage 4,N3,NREM 3 (slow-wave sleep),1,4
5,6,Sleep stage R,REM,Rapid eye movement,1,5
6,7,Movement time,MOVE,Movement time (unscored),0,6
7,8,Sleep stage ?,UNK,Unknown (unscored),0,7


## Consultas SQL

Las consultas están en `src/queries.py`. Comparten dos convenciones:

- `:include_artifacts` (0/1) controla si se incluyen las observaciones con artefacto;
- las comparaciones entre etapas usan solo etapas puntuadas (`is_scored = 1`). Las consultas de calidad (conteos y
  porcentaje de artefactos) describen todo el conjunto.

Como ejemplo, la consulta 1 completa:

In [ ]:
from src import queries

print(queries.QUERIES["sample_entropy_by_stage"])


        SELECT st.stage_code, c.short_name AS channel,
               COUNT(f.sample_entropy) AS n, AVG(f.sample_entropy) AS mean_sample_entropy,
               MIN(f.sample_entropy) AS min_value, MAX(f.sample_entropy) AS max_value
        
FROM fact_eeg_epoch f
JOIN dim_recording   r  ON r.recording_key  = f.recording_key
JOIN dim_channel     c  ON c.channel_key    = f.channel_key
JOIN dim_sleep_stage st ON st.stage_key     = f.stage_key
 WHERE st.is_scored = 1 AND (:include_artifacts = 1 OR f.is_artifact = 0)
        GROUP BY st.stage_code, st.stage_order, c.short_name
        ORDER BY st.stage_order, channel


In [ ]:
# 1. Sample Entropy promedio por etapa
q1 = queries.run_query("sample_entropy_by_stage")
display(q1.round(3))

,stage_code,channel,n,mean_sample_entropy,min_value,max_value
0,W,Fpz-Cz,11174,0.977,0.030,2.069
1,W,Pz-Oz,11233,1.291,0.041,2.038
2,N1,Fpz-Cz,4029,1.021,0.067,1.925
3,N1,Pz-Oz,4024,0.998,0.154,1.879
4,N2,Fpz-Cz,16635,0.798,0.062,1.674
5,N2,Pz-Oz,16619,0.816,0.154,1.856
6,N3,Fpz-Cz,2610,0.556,0.274,1.175
7,N3,Pz-Oz,2608,0.596,0.134,1.554
8,REM,Fpz-Cz,6849,0.996,0.083,1.874
9,REM,Pz-Oz,6846,1.007,0.106,1.608


In [ ]:
# 2. Spectral Entropy promedio por etapa
q2 = queries.run_query("spectral_entropy_by_stage")
display(q2.round(3))

,stage_code,channel,n,mean_spectral_entropy,min_value,max_value
0,W,Fpz-Cz,11174,0.604,0.258,0.918
1,W,Pz-Oz,11233,0.693,0.234,0.918
2,N1,Fpz-Cz,4029,0.624,0.271,0.886
3,N1,Pz-Oz,4024,0.649,0.260,0.887
4,N2,Fpz-Cz,16635,0.588,0.263,0.873
5,N2,Pz-Oz,16619,0.614,0.277,0.858
6,N3,Fpz-Cz,2610,0.486,0.338,0.671
7,N3,Pz-Oz,2608,0.519,0.331,0.719
8,REM,Fpz-Cz,6849,0.611,0.245,0.890
9,REM,Pz-Oz,6846,0.656,0.236,0.851


In [ ]:
# 3. Potencia relativa de cada banda por etapa
q3 = queries.run_query("relative_band_power_by_stage")
display(q3.round(3))

,stage_code,channel,n,delta,theta,alpha,sigma,beta
0,W,Fpz-Cz,11174,0.732,0.084,0.071,0.027,0.086
1,W,Pz-Oz,11233,0.499,0.125,0.179,0.054,0.143
2,N1,Fpz-Cz,4029,0.714,0.119,0.073,0.033,0.062
3,N1,Pz-Oz,4024,0.636,0.177,0.107,0.031,0.049
4,N2,Fpz-Cz,16635,0.800,0.101,0.041,0.038,0.020
5,N2,Pz-Oz,16619,0.746,0.146,0.052,0.037,0.020
6,N3,Fpz-Cz,2610,0.919,0.050,0.016,0.010,0.005
7,N3,Pz-Oz,2608,0.875,0.085,0.022,0.012,0.006
8,REM,Fpz-Cz,6849,0.739,0.128,0.056,0.026,0.050
9,REM,Pz-Oz,6846,0.656,0.178,0.095,0.031,0.041


In [ ]:
# 4. Comparación entre canales
q4 = queries.run_query("channel_comparison")
display(q4.round(3))

,channel,stage_code,n,sample_entropy,spectral_entropy,delta_rel,alpha_rel,total_power,rms
0,Fpz-Cz,W,11174,0.977,0.604,0.732,0.071,466.499,20.422
1,Pz-Oz,W,11233,1.291,0.693,0.499,0.179,82.858,8.443
2,Fpz-Cz,N1,4029,1.021,0.624,0.714,0.073,114.317,10.444
3,Pz-Oz,N1,4024,0.998,0.649,0.636,0.107,41.058,5.974
4,Fpz-Cz,N2,16635,0.798,0.588,0.800,0.041,180.271,13.271
5,Pz-Oz,N2,16619,0.816,0.614,0.746,0.052,66.968,7.958
6,Fpz-Cz,N3,2610,0.556,0.486,0.919,0.016,653.130,25.776
7,Pz-Oz,N3,2608,0.596,0.519,0.875,0.022,215.059,14.168
8,Fpz-Cz,REM,6849,0.996,0.611,0.739,0.056,94.034,9.590
9,Pz-Oz,REM,6846,1.007,0.656,0.656,0.095,32.695,5.622


In [ ]:
# 5. Comparación entre sujetos
q5 = queries.run_query("subject_comparison")
display(q5.round(3).head(20))

,subject_id,age,sex,stage_code,channel,n,sample_entropy,spectral_entropy,delta_rel
0,4,34,F,W,Fpz-Cz,385,0.837,0.529,0.832
1,4,34,F,W,Pz-Oz,385,1.296,0.677,0.574
2,4,34,F,N1,Fpz-Cz,303,1.266,0.637,0.670
3,4,34,F,N1,Pz-Oz,302,1.128,0.669,0.609
4,4,34,F,N2,Fpz-Cz,1134,0.934,0.638,0.703
5,4,34,F,N2,Pz-Oz,1134,0.935,0.667,0.660
6,4,34,F,N3,Fpz-Cz,147,0.614,0.514,0.881
7,4,34,F,N3,Pz-Oz,147,0.616,0.526,0.874
8,4,34,F,REM,Fpz-Cz,466,1.098,0.624,0.709
9,4,34,F,REM,Pz-Oz,466,1.088,0.681,0.615


In [ ]:
# 6. Cantidad de epochs por etapa
q6 = queries.run_query("epochs_by_stage")
display(q6.round(3))

,stage_code,n_epochs,percent,hours
0,W,11261,27.07,93.84
1,N1,4030,9.69,33.58
2,N2,16641,40.01,138.68
3,N3,2610,6.27,21.75
4,REM,6855,16.48,57.13
5,MOVE,39,0.09,0.33
6,UNK,160,0.38,1.33


In [ ]:
# 7. Porcentaje de epochs marcados como artefactos
q7 = queries.run_query("artifact_percentage")
display(q7.round(3))

,recording_id,channel,n_observations,n_artifacts,artifact_percent,n_stat_outliers,stat_outlier_percent
0,SC4041,Fpz-Cz,1236,0,0.00,110,8.90
1,SC4041,Pz-Oz,1236,0,0.00,156,12.62
2,SC4042,Fpz-Cz,1204,0,0.00,92,7.64
3,SC4042,Pz-Oz,1204,1,0.08,156,12.96
4,SC4051,Fpz-Cz,672,2,0.30,57,8.48
...,...,...,...,...,...,...,...
67,SC4732,Pz-Oz,1747,0,0.00,164,9.39
68,SC4761,Fpz-Cz,1703,0,0.00,175,10.28
69,SC4761,Pz-Oz,1703,0,0.00,228,13.39
70,SC4762,Fpz-Cz,1476,0,0.00,262,17.75


In [ ]:
# 8. Evolución temporal durante la noche
q8 = queries.run_query("temporal_evolution")
display(q8.round(3).head(20))

,recording_id,channel,hour_bin,n,sample_entropy,spectral_entropy,delta_rel,pct_n3,pct_rem
0,SC4041,Fpz-Cz,-1,60,0.805,0.502,0.876,0.0,0.0
1,SC4041,Fpz-Cz,0,120,1.121,0.645,0.681,0.8,0.0
2,SC4041,Fpz-Cz,1,120,0.832,0.584,0.777,29.2,12.5
3,SC4041,Fpz-Cz,2,120,0.955,0.629,0.698,0.0,21.7
4,SC4041,Fpz-Cz,3,120,0.804,0.598,0.752,13.3,0.0
5,SC4041,Fpz-Cz,4,119,1.056,0.631,0.690,0.0,30.3
6,SC4041,Fpz-Cz,5,120,0.914,0.623,0.716,0.8,0.0
7,SC4041,Fpz-Cz,6,120,1.105,0.663,0.662,0.0,20.8
8,SC4041,Fpz-Cz,7,120,1.099,0.628,0.696,0.0,50.0
9,SC4041,Fpz-Cz,8,120,1.111,0.647,0.680,0.0,8.3


In [ ]:
# Reproducibilidad: todas las consultas devuelven exactamente lo mismo al repetirse.
for name in queries.QUERIES:
    pd.testing.assert_frame_equal(queries.run_query(name), queries.run_query(name))
print(f"✔ {len(queries.QUERIES)} consultas reproducibles")

✔ 13 consultas reproducibles


> **Prompt 14.** *Escribí en SQL el cubo etapa × grupo etario × canal, con un slice de N2 y un roll-up de sujeto a grupo etario, y dibujá la figura del cubo con esos valores.*

## Cubo

Las 3 dimensiones del almacén forman un cubo de **5 etapas × 3 grupos etarios × 2 canales = 30 celdas**. Cada celda
resume las observaciones de esa combinación. SQLite no tiene un operador `CUBE`, así que cada operación es una consulta
con `GROUP BY` y `WHERE`.

| Operación | Qué hace | Consulta | Para qué se usa |
|---|---|---|---|
| **Slice** | Fija una dimensión | `slice_n2` | Pregunta 2: la etapa N2 según la edad |
| **Dice** | Restringe varias dimensiones | `dice_deep_sleep_extremes` | N2 y N3 en los grupos extremos |
| **Roll-up** | Sube en una jerarquía o agrega una dimensión | `rollup_stage` | Pregunta 1: el promedio por etapa |
| **Drill-down** | Baja en una jerarquía | `drilldown_n2_subject` | Del grupo etario al sujeto dentro de N2 |

La figura muestra el cubo con la **potencia relativa sigma**. La cara frontal es el canal Fpz-Cz y el corte resaltado
es el slice de N2. Es un esquema del modelo: los valores se comparan en las tablas y en la sección EDA.

In [ ]:
cube = queries.run_query("cube")
print(f"Celdas del cubo: {len(cube)} · sujetos por celda: {cube['n_subjects'].min()}–{cube['n_subjects'].max()}")
plots.cube_figure(cube, "sigma_rel", "N2").show()

Celdas del cubo: 30 · sujetos por celda: 5–6


In [ ]:
print("Slice: etapa = N2")
display(queries.run_query("slice_n2").round(3))
print("Dice: N2 y N3, grupos extremos")
display(queries.run_query("dice_deep_sleep_extremes").round(3))
print("Roll-up: solo la etapa")
display(queries.run_query("rollup_stage").round(3))
print("Drill-down: del grupo etario al sujeto, dentro de N2")
drilldown = queries.run_query("drilldown_n2_subject")
display(drilldown.round(3).head(20))

Slice: etapa = N2


,age_group,channel,n,n_subjects,sample_entropy,spectral_entropy,delta_rel,sigma_rel
0,<40,Fpz-Cz,5765,6,0.829,0.614,0.761,0.059
1,<40,Pz-Oz,5754,6,0.822,0.620,0.736,0.056
2,40-64,Fpz-Cz,5888,6,0.842,0.606,0.793,0.037
3,40-64,Pz-Oz,5892,6,0.826,0.623,0.744,0.032
4,65+,Fpz-Cz,4982,6,0.709,0.536,0.852,0.017
5,65+,Pz-Oz,4973,6,0.798,0.596,0.760,0.021


Dice: N2 y N3, grupos extremos


,stage_code,age_group,channel,n,n_subjects,sample_entropy,spectral_entropy,delta_rel,sigma_rel
0,N2,<40,Fpz-Cz,5765,6,0.829,0.614,0.761,0.059
1,N2,<40,Pz-Oz,5754,6,0.822,0.620,0.736,0.056
2,N2,65+,Fpz-Cz,4982,6,0.709,0.536,0.852,0.017
3,N2,65+,Pz-Oz,4973,6,0.798,0.596,0.760,0.021
4,N3,<40,Fpz-Cz,1281,6,0.570,0.501,0.915,0.013
5,N3,<40,Pz-Oz,1279,6,0.566,0.505,0.901,0.014
6,N3,65+,Fpz-Cz,723,5,0.523,0.450,0.931,0.006
7,N3,65+,Pz-Oz,723,5,0.599,0.522,0.852,0.009


Roll-up: solo la etapa


,stage_code,n,n_subjects,sample_entropy,spectral_entropy,delta_rel,sigma_rel
0,W,22407,18,1.135,0.648,0.615,0.040
1,N1,8053,18,1.009,0.637,0.675,0.032
2,N2,33254,18,0.807,0.601,0.773,0.038
3,N3,5218,16,0.576,0.503,0.897,0.011
4,REM,13695,18,1.001,0.634,0.698,0.028


Drill-down: del grupo etario al sujeto, dentro de N2


,age_group,subject_id,age,sex,channel,n,n_subjects,sample_entropy,spectral_entropy,delta_rel,sigma_rel
0,<40,10,26,M,Fpz-Cz,1278,1,0.753,0.590,0.813,0.049
1,<40,10,26,M,Pz-Oz,1278,1,0.799,0.593,0.799,0.038
2,<40,14,27,M,Fpz-Cz,790,1,0.876,0.646,0.745,0.063
3,<40,14,27,M,Pz-Oz,790,1,0.850,0.655,0.707,0.065
4,<40,5,28,F,Fpz-Cz,833,1,0.815,0.629,0.735,0.046
5,<40,5,28,F,Pz-Oz,833,1,0.773,0.624,0.691,0.046
6,<40,6,31,F,Fpz-Cz,823,1,0.784,0.624,0.762,0.050
7,<40,6,31,F,Pz-Oz,824,1,0.753,0.612,0.759,0.042
8,<40,16,32,M,Fpz-Cz,907,1,0.820,0.570,0.799,0.069
9,<40,16,32,M,Pz-Oz,895,1,0.797,0.574,0.787,0.077


## EDA

Todas las figuras se construyen con datos del **Data Warehouse** (vía `src/queries.py`), no desde los EDF. Cada una
responde una pregunta concreta.

**Paleta**: W naranja; N1→N3 una rampa azul de claro a oscuro (profundidad creciente del sueño); REM aqua. Se validó
para visión normal y daltonismo (protanopía/deuteranopía) en todos los pares de colores.

In [ ]:
obs = queries.load_observations()
print(f"Observaciones cargadas desde el DW: {len(obs):,}")

Observaciones cargadas desde el DW: 83,192


### Pregunta 1: ¿cómo varía el EEG entre etapas?

**¿Cuánto tiempo pasa en cada etapa dentro de la ventana de análisis?**

In [ ]:
plots.stage_distribution(queries.run_query("epochs_by_stage")).show()

**¿Cómo cambia la Sample Entropy entre etapas, y es igual en ambos canales?**

In [ ]:
plots.metric_by_stage(obs[~obs["is_artifact"]], "sample_entropy").show()

**¿Cómo cambia la Spectral Entropy entre etapas?**

In [ ]:
plots.metric_by_stage(obs[~obs["is_artifact"]], "spectral_entropy").show()

**¿Qué bandas dominan cada etapa? (canal posterior, donde el alfa de vigilia es más visible)**

In [ ]:
rel = queries.run_query("relative_band_power_by_stage")
plots.band_power_by_stage(rel, "Pz-Oz").show()

**¿En qué etapa es cada banda relativamente más fuerte? El color es el z-score dentro de cada banda; el texto, la potencia relativa media.**

In [ ]:
for ch in ["Fpz-Cz", "Pz-Oz"]:
    plots.stage_band_heatmap(rel, ch).show()

**¿Se asocian la actividad delta y la complejidad? ¿La relación depende de la etapa?**

In [ ]:
clean = obs[~obs["is_artifact"]]
for ch in ["Fpz-Cz", "Pz-Oz"]:
    plots.delta_vs_entropy(clean, ch, max_points_per_stage=1500).show()

**¿Difieren el canal frontal y el posterior?**

In [ ]:
plots.channel_comparison(queries.run_query("channel_comparison"), "sample_entropy").show()
plots.channel_comparison(queries.run_query("channel_comparison"), "alpha_rel").show()

**¿El patrón entre etapas se repite en cada sujeto, o lo produce un sujeto en particular?**

In [ ]:
plots.subject_comparison(queries.run_query("subject_comparison"), "sample_entropy", "Pz-Oz").show()

**¿Cómo evolucionan la delta relativa y la complejidad a lo largo de la noche?**

In [ ]:
evo = queries.run_query("temporal_evolution")
plots.temporal_evolution(evo, "delta_rel", "Fpz-Cz").show()
plots.temporal_evolution(evo, "sample_entropy", "Fpz-Cz").show()

**¿Cuán fuerte es la relación lineal entre las métricas, y cuánto de ella se explica por la etapa?** Se compara la
correlación de Spearman global con la mediana de las correlaciones dentro de cada etapa: si la correlación global es
mucho mayor, gran parte de la asociación se debe a que ambas variables cambian con la etapa (confusión), no a una
relación dentro de la etapa.

In [ ]:
pairs = [("delta_rel", "sample_entropy"), ("delta_rel", "spectral_entropy"), ("sample_entropy", "spectral_entropy")]
scored = obs[obs["is_scored"] & ~obs["is_artifact"]]
rows = []
for ch, g in scored.groupby("channel"):
    for a, b in pairs:
        within = g.groupby("stage_code").apply(lambda s: s[a].corr(s[b], method="spearman"), include_groups=False)
        rows.append({"canal": ch, "par": f"{a} ~ {b}", "ρ global": g[a].corr(g[b], method="spearman"),
                     "ρ dentro de etapa (mediana)": within.median()})
display(pd.DataFrame(rows).round(2))

,canal,par,ρ global,ρ dentro de etapa (mediana)
0,Fpz-Cz,delta_rel ~ sample_entropy,-0.88,-0.86
1,Fpz-Cz,delta_rel ~ spectral_entropy,-0.94,-0.92
2,Fpz-Cz,sample_entropy ~ spectral_entropy,0.84,0.84
3,Pz-Oz,delta_rel ~ sample_entropy,-0.85,-0.76
4,Pz-Oz,delta_rel ~ spectral_entropy,-0.91,-0.90
5,Pz-Oz,sample_entropy ~ spectral_entropy,0.86,0.85


### Pregunta 2: ¿cómo cambia el perfil con la edad?

**¿Cambia la actividad sigma de N2 con la edad?** La banda sigma (12–16 Hz) contiene los husos de sueño, que son
característicos de N2. Cada punto es un sujeto.

In [ ]:
plots.sigma_vs_age(drilldown).show()
for channel, group in drilldown.groupby("channel"):
    print(f"Spearman edad ~ sigma en N2, {channel}: ρ = {group['age'].corr(group['sigma_rel'], method='spearman'):.2f} "
          f"(n = {len(group)} sujetos)")

Spearman edad ~ sigma en N2, Fpz-Cz: ρ = -0.86 (n = 18 sujetos)
Spearman edad ~ sigma en N2, Pz-Oz: ρ = -0.76 (n = 18 sujetos)


**¿Cambia con la edad la distribución de la noche?** Porcentaje de épocas por etapa de cada sujeto, ordenados por edad.

In [ ]:
epochs_scored = obs[obs["is_scored"]].drop_duplicates(["recording_id", "epoch_number"])
architecture = (epochs_scored.groupby(["age", "subject_id", "stage_code"]).size().unstack(fill_value=0)
                .pipe(lambda d: d.div(d.sum(axis=1), axis=0).mul(100))[plots.STAGE_ORDER])
display(architecture.round(1))

stage_code         W    N1    N2    N3   REM
age subject_id                              
26  10          13.6   8.3  58.2   1.4  18.5
27  14          18.8   2.9  40.4  15.1  22.8
28  5           22.4   8.2  43.4  13.0  12.9
31  6           18.0   7.9  44.3  14.3  15.5
32  16          18.4   4.5  42.2  13.6  21.2
34  4           15.8  12.4  46.6   6.0  19.1
51  20          23.4   7.8  49.8   0.2  18.8
    38          15.2   8.5  57.8   2.6  15.9
54  24          23.7   7.0  49.4   2.4  17.5
    31          30.0  10.3  31.1  13.6  14.9
56  22          15.4  14.1  43.4   8.9  18.2
60  33          21.8  13.6  49.7   0.0  15.0
67  40          29.0  11.9  35.2   6.8  17.0
71  50          34.5  11.0  36.8   1.8  15.8
88  65          32.7   8.4  37.4   9.7  11.8
90  76          55.8   8.0  21.0   0.9  14.2
97  73          52.7  16.3  23.0   0.0   8.0
101 61          17.1  10.1  35.0   9.4  28.4

- **La sigma relativa de N2 cae con la edad**, de ≈0.057 en los menores de 40 a ≈0.019 en los mayores de 65, en ambos
  canales (Spearman −0.86 en Fpz-Cz y −0.76 en Pz-Oz, con 18 sujetos). Sin los dos sujetos más longevos (97 y 101 años)
  el resultado se mantiene (−0.82 con el promedio de los canales). Coincide con la disminución de husos de sueño que
  describe la literatura.
- **El sujeto de 71 años es la excepción:** tiene valores similares a los de 50 años.
- **La vigilia dentro de la ventana nocturna aumenta:** en promedio, 18 % en <40, 22 % en 40-64 y 37 % en 65+. Hay
  mucha variación: va de 17 % en el sujeto de 101 años a 56 % en el de 90.
- **Son 18 sujetos, 6 por grupo.** Es una tendencia descriptiva: no se puede separar la edad de las diferencias
  individuales ni estimar el tamaño del efecto.

> **Prompt 15.** *Hacé un EDA con Plotly que lea del almacén, organizado por las dos preguntas. Cada gráfico tiene que responder algo concreto.*

> **Prompt 16.** *Hacé un dashboard en Streamlit que lea solo del almacén y responda las dos preguntas. Cada sección tiene que tener un título que diga la conclusión, no el nombre de la variable.*

## Dashboard

`src/app.py` es un dashboard Streamlit que lee **solo** `data/database/sleep_dw.sqlite`.

- **Filtros**: grupo etario, sujeto, noche/registro, canal, etapas de sueño, incluir/excluir artefactos.
- **Pregunta 1**: Sample Entropy y Spectral Entropy por etapa, potencia relativa por banda y relación entre delta y
  complejidad.
- **Pregunta 2**: métrica según la edad de cada sujeto, con selectores de etapa y métrica para hacer otros cortes del
  cubo. Incluye el hipnograma de cada registro.

Ejecución local, desde la raíz del repositorio:

```bash
streamlit run src/app.py
```

En Colab, Streamlit necesita un túnel para exponer el puerto; las mismas figuras ya se muestran en la sección EDA de
este notebook. Ver el README para la opción con túnel.

## Presentación

> **Prompt 17.** *Para las diapositivas quiero un mensaje por gráfico: el título es la conclusión, todo en gris salvo lo importante, sin grilla ni bordes, y la etiqueta al lado de la serie.*

Los gráficos de este notebook son detallados, porque el informe se lee a su propio ritmo. Las diapositivas usan versiones
reducidas, con un solo mensaje por gráfico.

## Resultados principales

**Muestra:**
- 18 sujetos (9 mujeres y 9 hombres, de 26 a 101 años) × 2 noches = 36 registros;
- 41 596 épocas analizadas (≈347 h) y 83 192 observaciones época × canal.

Los valores citados corresponden a esta ejecución y se recalculan en las celdas anteriores. **Todos los resultados
son descriptivos.**

### Pregunta 1: etapas

**1. Distribución de etapas.** Dentro de la ventana nocturna, N2 es la etapa más frecuente (≈40 % de las épocas),
seguida de W (≈27 %, que incluye los despertares nocturnos), REM (≈16 %), N1 (≈10 %) y N3 (≈6 %).

- El N3 promedia ≈11 % en los menores de 40 y ≈5 % en los otros dos grupos, en línea con la disminución del sueño de
  ondas lentas con la edad que describe la literatura. La variación individual es grande: 1 % en un sujeto de 26 años y
  9 % en una de 101. Dos sujetos (60 y 97 años) no tienen ninguna época N3.
- Hubo 160 épocas `?` y 39 de *Movement time* dentro de la ventana. Se conservaron sin imputar.

**2. Complejidad temporal (Sample Entropy).**

- Baja con la profundidad del sueño NREM, N1 > N2 > N3 (en Pz-Oz ≈1.00 → 0.82 → 0.60). REM queda en niveles
  similares a N1 (≈1.01).
- El patrón es robusto entre sujetos: con promedios, **N1 > N2 > N3 se cumple en las 32 combinaciones sujeto × canal
  con N3**, y REM > N2 en las 36. Con medianas hay pocas excepciones: N1 > N2 > N3 falla en el sujeto 22 en Pz-Oz (31 de
  32), y REM > N2 en el sujeto 10 en Fpz-Cz y el 76 en Pz-Oz (34 de 36).
- La vigilia muestra la mayor complejidad en Pz-Oz (≈1.29). En Fpz-Cz es menor y más variable, compatible con la
  contaminación por movimientos oculares, que agrega actividad lenta y regular en el canal frontal.

**3. Dispersión espectral (Spectral Entropy).** N3 tiene la menor Spectral Entropy (≈0.49–0.52) en **las 32
combinaciones sujeto × canal con N3** usando promedios (31 de 32 con medianas; la excepción es de nuevo el sujeto 22 en
Pz-Oz): la potencia se concentra en pocas frecuencias. Las dos entropías cuentan
historias parecidas pero no idénticas, porque miden propiedades distintas de la señal.

**4. Actividad espectral.**

- La **delta relativa** crece con la profundidad del sueño: ≈0.87–0.92 en N3.
- El **alfa relativo** es máximo en vigilia en el canal posterior (Pz-Oz ≈0.18, contra ≈0.07 en Fpz-Cz), coherente
  con el ritmo alfa occipital de la vigilia relajada.
- **Theta** es relativamente más alta en N1 y REM.
- **Beta** es máxima en vigilia.

**5. Delta y complejidad.** La delta relativa y la Sample Entropy tienen una fuerte asociación negativa (Spearman
ρ ≈ −0.86). La correlación se mantiene **dentro de cada etapa** (ρ ≈ −0.66 a −0.91, salvo W en Pz-Oz con ≈ −0.49),
así que no se explica solo por el cambio de etapa: dentro de un mismo estado, las épocas con más potencia lenta son
más predecibles. Es una asociación entre dos descriptores de la misma señal, **no una relación causal**.

**6. Evolución nocturna.** El N3 se concentra en las primeras horas tras el inicio del sueño (≈10–18 % en las horas
0–2) y casi desaparece después de la hora 5. El REM, en cambio, aumenta hacia la segunda mitad de la noche (≈26–28 % en
las horas 5–6). Es el patrón clásico de la arquitectura del sueño. Al final de la noche quedan pocos registros (las
noches tienen duraciones distintas), por eso el gráfico solo muestra las horas que cubren al menos la mitad de ellos.

### Pregunta 2: edad

**7. La actividad sigma de N2 cae con la edad.** La potencia relativa sigma en N2 baja de ≈0.057 en los menores de 40
a ≈0.035 en el grupo de 40–64 y a ≈0.019 en los mayores de 65, unas 3 veces menos (Spearman −0.86 en Fpz-Cz y −0.76 en
Pz-Oz, con 18 sujetos). Es compatible con la disminución de husos de sueño con la edad. También aumenta la vigilia dentro
de la noche, de 18 % en <40 a 37 % en 65+. La tendencia es descriptiva, y hay excepciones, como el sujeto de 71 años en
sigma y la de 101 años en vigilia.

### Calidad de datos

**8. Calidad de datos.**

- **Artefactos físicos**: 173 observaciones (0.21 %) con saturación del conversor, 115 de ellas en W y solo 2 en N3. Los
  artefactos sutiles (oculares, musculares) no se detectan con estos criterios: es una limitación, no una prueba de
  ausencia de artefactos.
- **Outliers estadísticos**: 9.8 % de las observaciones según la regla de Tukey dentro de cada noche, entre 3.8 % y
  19.9 % por registro y canal. Son informativos y no se eliminan.
- **Comparar registros también es útil.** Con Tukey entre registros, SC4221 tiene 42 % de filas marcadas: esa regla no
  sirve para encontrar épocas raras, pero sí para detectar un registro distinto.
- **El caso SC4221 / Pz-Oz** muestra para qué sirve esa comparación. El canal posterior de esa noche tiene una amplitud
  ~4 veces menor que la otra noche del mismo sujeto (RMS mediana ≈1.9 µV contra ≈6.4 µV), mientras que Fpz-Cz es
  normal. Esto sugiere un problema técnico de registro (contacto del electrodo o ganancia) más que fisiología.
  - No llega a ser una señal plana, por lo que no se marca como artefacto.
  - Sus métricas **relativas** e invariantes a la escala (potencia relativa, Sample Entropy con r = 0.2·SD,
    Spectral Entropy) siguen siendo interpretables.
  - Sus métricas **absolutas** no lo son.
  - El diagnóstico no eliminó nada, pero permitió detectar el caso y documentarlo: un outlier estadístico es una
    señal para revisar, no un criterio de exclusión.

**En síntesis:** al profundizarse el sueño NREM, el EEG se vuelve más lento (más delta), más regular (menor Sample
Entropy) y con la potencia más concentrada (menor Spectral Entropy). REM y N1 se parecen a la vigilia en complejidad,
pero con menos alfa posterior. El patrón se repite en sujetos de 26 a 101 años. La edad no cambia ese orden, pero sí
algunos rasgos: cae la actividad sigma de N2, aumenta la vigilia y varía mucho la cantidad de sueño profundo. Estas observaciones coinciden con lo conocido sobre la fisiología del sueño, lo que funciona
también como validación del pipeline.

## Limitaciones

- **Muestra pequeña**: 18 sujetos sanos de un único estudio, estratificados por sexo y edad (26–101 años). Con 3 sujetos
  por estrato no se pueden separar los efectos de edad y sexo de la variabilidad individual. Los resultados no se
  generalizan a patologías.
- **Edad como fuente de variabilidad**: el sueño de ondas lentas disminuye con la edad, así que los promedios por
  etapa mezclan sujetos con arquitecturas de sueño muy distintas. La comparación entre sujetos lo hace visible.
- **Pregunta 2 con pocos sujetos**: cada grupo etario tiene 6 sujetos, y el grupo 65+ incluye edades extremas (97 y 101
  años). Los promedios del cubo pesan por cantidad de épocas, así que un sujeto con más épocas influye más en su celda.
- **Sin inferencia estadística**: las épocas de una misma noche no son independientes (autocorrelación temporal y
  anidamiento sujeto/noche). Un análisis inferencial necesitaría modelos mixtos con sujeto como efecto aleatorio.
- **Puntuación R&K**: los hipnogramas siguen Rechtschaffen & Kales (1968); unir S3+S4 en N3 aproxima AASM, pero
  los criterios no son idénticos.
- **Dos canales bipolares**: Fpz-Cz y Pz-Oz no permiten análisis topográfico. Fpz-Cz es sensible a la actividad
  ocular, que cae en la banda delta, sobre todo en vigilia.
- **Calidad por canal y registro**: no se evalúa automáticamente si un canal entero tiene una amplitud anómala. El
  caso SC4221 / Pz-Oz (amplitud ~4 veces menor que en la otra noche) se detectó al comparar las tasas de
  outliers entre registros y se documentó, pero sus métricas absolutas no son comparables con las del resto.
- **Detección de artefactos conservadora**: los criterios físicos (saturación, señal plana, no finitos) no detectan
  artefactos sutiles, como movimientos oculares o EMG de baja amplitud. No se aplicaron métodos avanzados (ICA,
  regresión de EOG) para evitar sobreprocesar la señal.
- **Convenciones de las métricas**: la potencia absoluta depende del método de integración, y la Sample Entropy
  depende de m, r y la longitud de la época. Las comparaciones son válidas dentro de este pipeline, no necesariamente
  con valores publicados que usen otros parámetros.
- **Recorte de vigilia**: conservar ±30 min es una convención. Otra ventana cambiaría la proporción de W y sus
  estadísticos.

## Reproducibilidad

| Aspecto | Cómo se garantiza |
|---|---|
| Datos | Descarga programática desde la fuente pública; verificación SHA256 contra los hashes oficiales; manifest |
| Rutas | `pathlib`, relativas a la raíz del repositorio; sin rutas absolutas ni archivos privados |
| Parámetros | Centralizados en `src/config.py`; su hash se guarda con cada resultado intermedio |
| Caché | Por registro, invalidado automáticamente si cambian parámetros o versión del código |
| Aleatoriedad | El submuestreo del scatter y los detectores de outliers usan `RANDOM_SEED` |
| Bases | Se reconstruyen desde cero en cada ejecución; surrogate keys determinísticas |
| Consultas | Verificadas como reproducibles (mismo resultado en ejecuciones repetidas) |
| Entorno | Versiones registradas en `data/processed/environment.json` |

In [ ]:
env = json.loads(pipeline.ENVIRONMENT_PATH.read_text())
display(pd.Series(env["library_versions"], name="versión").to_frame())
print("Hash de parámetros del Transform:", env["params_hash"])

,versión
python,3.13.11
numpy,2.2.6
pandas,2.2.3
scipy,1.15.3
mne,1.9.0
mne-bids,0.20.0
antropy,0.1.9
pyarrow,25.0.1
plotly,7.1.0
streamlit,1.65.0


Hash de parámetros del Transform: a9537c030c33


## Conclusiones

Se construyó un pipeline de Ingeniería de Datos reproducible de punta a punta:

- **Extract** desde una fuente pública, con integridad verificada.
- **Organización BIDS** que separa los datos crudos de su representación estandarizada.
- **Transform** documentado: épocas, recorte, filtro y features con librerías científicas mantenidas.
- **Control de calidad** explícito, que distingue datos faltantes, outliers estadísticos y artefactos de señal sin
  eliminar ni imputar información.
- **Modelado relacional 3NF**, con restricciones verificadas.
- **Data Warehouse en estrella con 3 dimensiones**, alimentado por un ETL desde el OLTP, que forma el cubo
  etapa × grupo etario × canal.
- **Consultas SQL reproducibles**, visualizaciones y un dashboard que leen únicamente del DW.

Sobre la **pregunta 1**, los datos muestran de forma consistente entre sujetos y canales que, a medida que el sueño NREM
se profundiza, la potencia se concentra en delta y la complejidad (en sus dos acepciones) disminuye. REM y N1 se parecen
más a la vigilia en complejidad, pero con menos alfa posterior que la vigilia relajada.

Sobre la **pregunta 2**, la actividad sigma de N2 cae con la edad y la vigilia nocturna aumenta. Con 18 sujetos, es una
tendencia que orienta, no una estimación del efecto.

Estas observaciones son descriptivas y coinciden con lo esperado por la fisiología del sueño, lo que funciona también
como validación del pipeline.

### Trabajo futuro: almacenamiento de EEG continuo

Este trabajo almacena **features por época**: un volumen pequeño que SQLite maneja sin dificultad. Si el objetivo
fuera almacenar **señal EEG continua**, por ejemplo con adquisición en tiempo real o *wearables* de sueño que
transmiten muestras de forma permanente, convendría una base de series temporales como **TimescaleDB** (extensión de
PostgreSQL), que ofrece:

- particionado automático por tiempo (*hypertables*);
- compresión columnar;
- agregados continuos (por ejemplo, potencia por banda cada 30 s calculada en la base);
- políticas de retención.

No es necesaria para este TP, cuyo foco es el modelado OLTP/DW de métricas derivadas.